# SUN397 TACS visual L/16: export existing best.pt, no retraining

This notebook recovers a complete web demo bundle from the saved **TACS visual checkpoint**, not from a new training run. It never calls training, constructs an optimizer, loads last.pt, downloads models or invents training_complete.json. It builds the backbone with pretrained=False and strictly loads all weights from best.pt, so the original DINO pretrained .pth is unnecessary.

Required: the original saved run folder with best.pt, protocol.json, class_names.json, candidate_pool.csv, train_split.csv and validation_split.csv; SUN397 images; and the original DINOv3 code folder with license. Keep embedded_runtime/ or embedded_runtime_v2/ when available. Known embedded source is only a fallback when its inference hash matches the checkpoint protocol. A checkpoint from a different architecture is rejected, not silently converted. Production requires ViT-L/16, 384px, K=1 and 397 classes.

No captions, Florence, CLIP or text vectors are needed. There is no separately trained selector or attention model: the complete shared visual backbone and classifier are loaded from best.pt.

**Time-saving defaults:** encode only the saved candidate pool once, do not resample it, and skip full test-set evaluation. RUN_FULL_TEST=False creates a clearly labeled checkpoint-snapshot demo; a one-image smoke test is not a test-set metric. Original test metrics, if present, are preserved as unverified reference files, not represented as a new evaluation. Enable RUN_FULL_TEST only when a new fp32 full-test run is desired.

The output directory is separate. Original weights, source, CSV, cache, pool and protocol are read-only. Interrupted exports retain verified model/bank stages for retry. A cold offline process reloads the bundle, checks predictions against best.pt, and computes both actual visual attention maps before ZIP publication. No extra attention model is required. The exported adapter is ready to be wired into the existing FastAPI web; this notebook does not modify that app.

Restart the kernel before running this notebook after a training notebook to avoid importing another run's modules. Only load trusted checkpoint/code files. No Internet or global Transformers downgrade is needed. Offline caption dependencies may be prepared in an isolated child environment for M-TACS. Kaggle Linux wheels are not Windows installers; the web host still needs compatible Python/PyTorch/dependencies. Keep original model/dataset license terms.


## 1 Offline preflight
No project Python upload is required: the exporter and known fallback source are embedded below.


In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
import sys
import importlib.util
from pathlib import Path
missing = [name for name in ("torch", "numpy", "PIL", "tqdm") if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError("Missing offline packages: " + ", ".join(missing))
import torch
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())


## 2 Point to the existing run, not a new training output

SOURCE_RUN_DIR must be the exact folder containing best.pt and its sidecars, not the Kaggle Input root and not a deploy ZIP. Edit YOUR_SAVED_RUN_FOLDER below after attaching the saved Output as an Input. You may also set it to the original /kaggle/working/outputs/... folder in the same session after restarting the kernel. SOURCE_RUN_DIR can be best.pt itself when its sidecars are in the same directory.

Attach the dataset and DINO source, but no DINO pretrained weights are needed. FEATURE_BATCH_SIZE=8 is an inference batch for candidate encoding, not a training batch. EXAMPLE_IMAGE=None selects a nonduplicate saved validation image; an explicit external image can be used for a demo smoke test. Leave RUN_FULL_TEST=False for the fastest export. Output must be outside the original run.


In [ ]:
VARIANT = 'tacs'
SOURCE_RUN_DIR = Path(os.getenv("SUN397_TACS_SAVED_RUN", "/kaggle/input/YOUR_SAVED_RUN_FOLDER"))
DATASET_ROOT = Path(os.getenv("SUN397_ROOT", "/kaggle/input/datasets/hiuphmnhtrung/sun397/SUN397/SUN397"))
DINOV3_REPO_DIR = Path(os.getenv("DINOV3_REPO_DIR", "/kaggle/input/datasets/sinhnguync/dinov3-code/dinov3-main"))
PROJECT_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
OUTPUT_DIR = Path(os.getenv("SUN397_TACS_EXPORT_ONLY_OUTPUT", str(PROJECT_DIR / "exports_sun397_tacs_visual_vitl16")))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FEATURE_BATCH_SIZE = 8
NUM_WORKERS = min(4, os.cpu_count() or 1)
EXAMPLE_IMAGE = None
RUN_FULL_TEST = False
print("Read-only source:", SOURCE_RUN_DIR)
print("New export output:", OUTPUT_DIR)
print("Device:", DEVICE, "| full test:", RUN_FULL_TEST)


## 3 Embedded exporter
Importing the original training helpers only defines functions; no training or full-caption generation function is called.


In [ ]:
EXPORT_CODE_DIR = OUTPUT_DIR / "notebook_exporter"
EXPORT_CODE_DIR.mkdir(parents=True, exist_ok=True)
EXPORTER_SOURCE = '"""Read-only checkpoint-to-web export. Never trains or invents completion markers."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport importlib\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport shutil\nimport subprocess\nimport sys\nimport zipfile\n\nimport torch\n\n\nVERSION = "sun397_snapshot_export_v1"\nINFERENCE_FIXES = (\n    ("vectors[~mask] = 0", "vectors = vectors.masked_fill((~mask).unsqueeze(1), 0.0)"),\n    ("vectors[~masks] = 0", "vectors = vectors.masked_fill((~masks).unsqueeze(1), 0.0)"),\n)\n\n\ndef sha(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for chunk in iter(lambda: handle.read(4 * 1024**2), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef json_read(path):\n    return json.loads(Path(path).read_text(encoding="utf-8-sig"))\n\n\ndef json_write(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef csv_read(path):\n    with Path(path).open(encoding="utf-8-sig", newline="") as handle:\n        rows = list(csv.DictReader(handle))\n    for row in rows:\n        row["class_id"] = int(row["class_id"])\n        pure = PurePosixPath(row["image_id"])\n        if pure.is_absolute() or ".." in pure.parts or "\\\\" in row["image_id"] or ":" in row["image_id"]:\n            raise ValueError("Unsafe image ID: " + row["image_id"])\n    if len({r["image_id"] for r in rows}) != len(rows):\n        raise ValueError("Duplicate IDs in " + str(path))\n    return rows\n\n\ndef fixed_caption_source(source):\n    for old, new in INFERENCE_FIXES:\n        source = source.replace(old, new)\n    compile(source, "caption_worker", "exec")\n    return source\n\n\ndef source_variants(source, caption=False):\n    variants = [source]\n    if caption:\n        variants.append(fixed_caption_source(source))\n        previous = source\n        for old, new in INFERENCE_FIXES:\n            previous = previous.replace(new, old)\n        variants.append(previous)\n    return variants\n\n\ndef original_identity(config, classes, frames, assets, source_hashes, variant):\n    ids = {key: [(r["image_id"], int(r["class_id"])) for r in rows] for key, rows in frames.items()}\n    if variant == "tacs":\n        model_sha, helper_sha = source_hashes["runtime"], source_hashes["training"]\n    else:\n        model_sha, helper_sha = source_hashes["tacs_visual_web"], source_hashes["train_sun397_tacs_web"]\n        assets = dict(assets, mtacs_sources=source_hashes)\n    payload = {"config": config, "classes": classes, "splits_and_pool": ids, "assets": assets,\n               "model_source_sha256": model_sha, "training_source_sha256": helper_sha}\n    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()\n\n\ndef resolve_image(record, root):\n    root = Path(root).expanduser().resolve()\n    pure = PurePosixPath(record["image_id"])\n    for prefix in (root, root / "SUN397", root / "images"):\n        path = prefix.joinpath(*pure.parts).resolve()\n        if path.is_relative_to(root) and path.is_file():\n            return dict(record, image_path=str(path))\n    raise FileNotFoundError(f"Cannot resolve {record[\'image_id\']} below DATASET_ROOT={root}")\n\n\ndef _runtime_sources(run, output, variant, fallback, recorded):\n    required = ["tacs_visual_web.py", "train_sun397_tacs_web.py"]\n    if variant == "mtacs":\n        required += ["sun397_caption_web.py", "mtacs_multimodal_web.py", "train_sun397_mtacs_web.py"]\n    critical = {"tacs_visual_web.py": recorded["runtime"] if variant == "tacs" else recorded["tacs_visual_web"]}\n    if variant == "mtacs":\n        critical.update({"sun397_caption_web.py": recorded["sun397_caption_web"],\n                         "mtacs_multimodal_web.py": recorded["mtacs_multimodal_web"]})\n    helper_hashes = {"train_sun397_tacs_web.py": recorded["training"] if variant == "tacs" else recorded["train_sun397_tacs_web"]}\n    if variant == "mtacs":\n        helper_hashes["train_sun397_mtacs_web.py"] = recorded["training"]\n    directory = output / "export_runtime"\n    directory.mkdir(parents=True, exist_ok=True)\n    provenance = {}\n    for name in required:\n        paths = [run / folder / name for folder in ("embedded_runtime_v2", "embedded_runtime")]\n        original = next((p for p in paths if p.is_file()), None)\n        content = original.read_text(encoding="utf-8") if original else fallback.get(name)\n        if content is None:\n            raise FileNotFoundError("Original runtime missing and no known fallback: " + name)\n        provided_sha = hashlib.sha256(content.encode()).hexdigest()\n        if original is not None and name in helper_hashes and hashlib.sha256(content.encode()).hexdigest() != helper_hashes[name]:\n            raise ValueError("Original helper source changed after training: " + name)\n        if name in critical:\n            matched = next((value for value in source_variants(content, name == "sun397_caption_web.py")\n                            if hashlib.sha256(value.encode()).hexdigest() == critical[name]), None)\n            if matched is None:\n                raise ValueError("Inference source differs from checkpoint protocol: " + name + ". Attach its original embedded_runtime folder.")\n            content = matched\n        input_sha = hashlib.sha256(content.encode()).hexdigest()\n        if name == "sun397_caption_web.py":\n            content = fixed_caption_source(content)\n        compile(content, name, "exec")\n        path = directory / name\n        if path.is_file() and path.read_text(encoding="utf-8") != content:\n            raise ValueError("Export runtime already belongs to another source; select a new OUTPUT_DIR")\n        if not path.is_file():\n            path.write_text(content, encoding="utf-8", newline="\\n")\n        loaded = sys.modules.get(Path(name).stem)\n        if loaded is not None and Path(loaded.__file__).resolve() != path.resolve():\n            raise RuntimeError("Another TACS runtime is imported; restart the kernel before running this export-only notebook")\n        provenance[name] = {"origin": str(original) if original else "embedded_known_fallback",\n                            "provided_source_sha256": provided_sha,\n                            "input_sha256": input_sha, "export_sha256": sha(path),\n                            "caption_inference_tensor_fix": input_sha != sha(path)}\n    sys.path.insert(0, str(directory.resolve()))\n    modules = {Path(name).stem: importlib.import_module(Path(name).stem) for name in required}\n    return modules, provenance\n\n\ndef prepare(source_run, dataset_root, dino_repo, output, variant, fallback_sources,\n            device="cuda", feature_batch_size=8, workers=0, example_image=None,\n            florence=None, clip=None, offline_assets_root=None,\n            run_full_test=False, model_factory=None, caption_runner=None):\n    if variant not in {"tacs", "mtacs"}:\n        raise ValueError("variant must be tacs or mtacs")\n    source = Path(source_run).expanduser().resolve()\n    best_path = source if source.is_file() else source / "best.pt"\n    run = best_path.parent\n    output = Path(output).expanduser().resolve()\n    if output == run or output.is_relative_to(run):\n        raise ValueError("Use a separate writable OUTPUT_DIR outside the original run")\n    required = ("protocol.json", "class_names.json", "candidate_pool.csv", "train_split.csv", "validation_split.csv")\n    for path in [best_path] + [run / name for name in required]:\n        if not path.is_file():\n            raise FileNotFoundError(f"Required saved artifact missing: {path}. Attach the original run folder, not best.pt alone.")\n    if feature_batch_size < 1 or workers < 0:\n        raise ValueError("Invalid inference batch/workers")\n    device = torch.device(device)\n    if device.type == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("CUDA requested but not available; enable a GPU or select cpu")\n    protocol = json_read(run / "protocol.json")\n    best_sha = sha(best_path)\n    checkpoint = torch.load(best_path, map_location="cpu", weights_only=True)\n    if not all(key in checkpoint for key in ("model_state_dict", "config", "class_names", "identity", "epoch")):\n        raise ValueError("Unsupported checkpoint schema; do not guess architecture or rename state_dict keys")\n    config = checkpoint["config"]\n    classes = checkpoint["class_names"]\n    expected_method = "TACS_visual" if variant == "tacs" else "M_TACS_full_both"\n    expected_schema = "sun397_tacs_visual_web_v1" if variant == "tacs" else "sun397_mtacs_full_both_web_v1"\n    if (config.get("method") != expected_method or config.get("schema") != expected_schema\n            or config.get("dataset") != "SUN397" or config.get("context_top_k") != 1\n            or bool(config.get("caption_used")) != (variant == "mtacs")):\n        raise ValueError("Checkpoint is not the requested SUN397 web architecture; original ablation checkpoints are not interchangeable")\n    if model_factory is None and (config.get("backbone") != "dinov3_vitl16" or config.get("feature_dim") != 1024\n                                  or config.get("image_size") != 384 or config.get("num_classes") != 397):\n        raise ValueError("Production export requires L/16, 1024 dimensions, 384px and 397 classes")\n    if config != protocol["config"] or classes != json_read(run / "class_names.json") or checkpoint["identity"] != protocol["identity"]:\n        raise ValueError("best.pt, protocol and class order do not belong to the same experiment")\n    if len(classes) != config["num_classes"] or len(set(classes)) != len(classes):\n        raise ValueError("Invalid authoritative class order")\n    frames = {key: csv_read(run / name) for key, name in\n              (("train", "train_split.csv"), ("val", "validation_split.csv"), ("pool", "candidate_pool.csv"))}\n    for rows in frames.values():\n        for row in rows:\n            if not 0 <= row["class_id"] < len(classes) or str(PurePosixPath(row["image_id"]).parent) != classes[row["class_id"]]:\n                raise ValueError("Saved image/class mapping does not match checkpoint class order")\n    pool_ids = {r["image_id"] for r in frames["pool"]}\n    if not pool_ids or not pool_ids <= {r["image_id"] for r in frames["train"]} or pool_ids & {r["image_id"] for r in frames["val"]}:\n        raise ValueError("Candidate pool must be the saved training-only pool")\n    if original_identity(config, classes, frames, protocol["assets"], protocol["source_sha256"], variant) != checkpoint["identity"]:\n        raise ValueError("Saved pool/order/splits/assets or source identity changed; do not regenerate a random pool")\n    for index, row in enumerate(frames["pool"]):\n        if "position" in row and int(row["position"]) != index:\n            raise ValueError("Candidate positions are not in their trained order")\n    output.mkdir(parents=True, exist_ok=True)\n    modules, source_provenance = _runtime_sources(run, output, variant, fallback_sources, protocol["source_sha256"])\n    visual, common = modules["tacs_visual_web"], modules["train_sun397_tacs_web"]\n    dino_repo = Path(dino_repo).expanduser().resolve()\n    if common.source_tree_identity(dino_repo) != protocol["assets"].get("dino_source_sha256"):\n        raise ValueError("DINOv3 source differs from the recorded training source; attach the original code folder")\n    candidates = [resolve_image(row, dataset_root) for row in frames["pool"]]\n    candidate_hashes = {row["image_id"]: sha(row["image_path"]) for row in candidates}\n    if sha(best_path) != best_sha:\n        raise ValueError("best.pt changed while being read; stop concurrent training and export a stable snapshot")\n    complete_path = run / "training_complete.json"\n    completion = "not_recorded; export of checkpoint snapshot only"\n    if complete_path.is_file():\n        marker = json_read(complete_path)\n        if (marker["identity"] != checkpoint["identity"] or marker["best_sha256"] != best_sha\n                or marker["best_epoch"] != checkpoint["epoch"] or marker.get("epochs_completed") != config["epochs"]):\n            raise ValueError("Existing completion marker does not match best.pt")\n        completion = "original completion marker verified"\n    context = {"variant": variant, "run": run, "best_path": best_path, "best_sha": best_sha,\n               "checkpoint": checkpoint, "config": config, "classes": classes, "protocol": protocol,\n               "frames": frames, "candidates": candidates, "candidate_hashes": candidate_hashes, "dataset_root": Path(dataset_root),\n               "dino_repo": dino_repo, "output": output, "device": device, "modules": modules,\n               "source_provenance": source_provenance, "feature_batch_size": feature_batch_size, "workers": workers,\n               "example_image": Path(example_image).resolve() if example_image else None,\n               "run_full_test": bool(run_full_test), "model_factory": model_factory, "caption_runner": caption_runner,\n               "completion": completion, "offline_assets_root": Path(offline_assets_root) if offline_assets_root else None}\n    if variant == "mtacs":\n        service, workflow = modules["sun397_caption_web"], modules["train_sun397_mtacs_web"]\n        shared = run / "shared"\n        for name in ("captions.csv", "text_embeddings.pt", "caption_protocol.json", "caption_settings.json"):\n            if not (shared / name).is_file():\n                raise FileNotFoundError("M-TACS export requires the ORIGINAL shared cache: " + str(shared / name))\n        for key, name in (("caption_csv_sha256", "captions.csv"), ("text_cache_sha256", "text_embeddings.pt"),\n                          ("caption_settings_sha256", "caption_settings.json")):\n            if sha(shared / name) != protocol["assets"].get(key):\n                raise ValueError("Original caption cache binding changed: " + name)\n        cache = torch.load(shared / "text_embeddings.pt", map_location="cpu", weights_only=True)\n        records = frames["train"] + frames["val"]\n        test_path = run / "test_split.csv"\n        if not test_path.is_file():\n            raise FileNotFoundError("M-TACS full caption coverage also requires test_split.csv")\n        test = csv_read(test_path)\n        records += test\n        table = workflow.TextTable(cache, shared / "captions.csv", records, config)\n        settings = json_read(shared / "caption_settings.json")\n        if settings["classes"] != classes or settings["settings"]["policy"] != config["caption_policy"]:\n            raise ValueError("Caption settings/class list differs from trained text table")\n        if model_factory is None:\n            if not florence or not clip:\n                raise ValueError("Attach complete frozen Florence and CLIP folders for novel-image inference")\n            assets = service.validate_assets(florence, clip)\n        else:\n            # Tests inject a tiny visual model and explicit caption service only.\n            assets = {"florence": str(florence), "clip": str(clip),\n                      "florence_sha256": service.tree_identity(florence), "clip_sha256": service.tree_identity(clip)}\n        for key in ("florence_sha256", "clip_sha256"):\n            if assets[key] != settings["asset_identity"][key] or assets[key] != protocol["assets"]["caption_assets"][key]:\n                raise ValueError("Frozen text model weights differ from the caption cache: " + key)\n        for row in candidates:\n            if candidate_hashes[row["image_id"]] != table.rows[row["image_id"]]["image_sha256"]:\n                raise ValueError("Candidate image bytes differ from its registered caption: " + row["image_id"])\n        context.update(table=table, model_assets=assets, caption_settings=settings, shared=shared)\n        if caption_runner is None:\n            service.caption_environment(output / "caption_preflight", context["offline_assets_root"])\n    history_path = run / "history.csv"\n    last_saved = None\n    if history_path.is_file():\n        with history_path.open(encoding="utf-8-sig", newline="") as handle:\n            history = list(csv.DictReader(handle))\n        last_saved = history[-1].get("epoch") if history else None\n    context["last_saved_epoch_from_history"] = last_saved\n    print(f"Snapshot: {variant}, best epoch={checkpoint[\'epoch\']}, history last epoch={last_saved}, pool={len(candidates)}")\n    print("Completion:", completion, "| Full test:", "enabled" if run_full_test else "not run; demo snapshot export")\n    print("No optimizer, pretrained DINO checkpoint, training loop or full caption regeneration is needed")\n    return context\n\n\ndef _copy_file(source, target):\n    source, target = Path(source), Path(target)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    if target.is_file():\n        if sha(source) != sha(target):\n            raise ValueError("Partial export file differs from source: " + str(target))\n        return\n    temporary = target.with_name(target.name + ".tmp")\n    shutil.copy2(source, temporary)\n    temporary.replace(target)\n\n\ndef _copy_assets(source, target, service):\n    source, target = Path(source), Path(target)\n    for path in sorted(source.rglob("*")):\n        relative = path.relative_to(source)\n        if path.is_file() and not {".cache", ".git", "__pycache__"}.intersection(relative.parts) and path.suffix != ".pyc":\n            _copy_file(path, target / relative)\n    if service.tree_identity(source) != service.tree_identity(target):\n        raise ValueError("Frozen model copy is incomplete or has unexpected files")\n\n\ndef _progress(context, name, create):\n    """Reuse completed expensive stages only after checking their recorded hashes."""\n    stage = context["stage"]\n    progress_path = stage / "export_progress.json"\n    progress = json_read(progress_path) if progress_path.is_file() else {"files": {}}\n    path = stage / name\n    if name in progress["files"]:\n        if not path.is_file() or sha(path) != progress["files"][name]:\n            raise ValueError("Staged export artifact changed: " + name)\n        print("Reusing verified export artifact:", name)\n        return path\n    if path.exists():\n        raise FileExistsError("Unregistered partial artifact; preserve this output and use a new OUTPUT_DIR: " + name)\n    create(path)\n    progress["files"][name] = sha(path)\n    json_write(progress_path, progress)\n    return path\n\n\ndef _manifest(context):\n    stage = context["stage"]\n    files = {path.relative_to(stage).as_posix(): {"bytes": path.stat().st_size, "sha256": sha(path)}\n             for path in sorted(stage.rglob("*")) if path.is_file() and path.name != "manifest.json"\n             and "__pycache__" not in path.parts and not path.name.endswith(".tmp")}\n    json_write(stage / "manifest.json", {"schema": context["runtime"].SCHEMA, "status": "ready", "files": files,\n                                        "run_identity": context["checkpoint"]["identity"],\n                                        "training_checkpoint_sha256": context["best_sha"],\n                                        "model_sha256": sha(stage / "model.pt"),\n                                        "operation": "export_only_snapshot", "export_request": context["request_identity"]})\n\n\ndef build(context):\n    ctx = context\n    variant, output, modules = ctx["variant"], ctx["output"], ctx["modules"]\n    visual, common = modules["tacs_visual_web"], modules["train_sun397_tacs_web"]\n    runtime = visual if variant == "tacs" else modules["mtacs_multimodal_web"]\n    ctx["runtime"] = runtime\n    request = {"version": VERSION, "operation": "export_only_snapshot", "variant": variant,\n               "best_sha256": ctx["best_sha"], "run_identity": ctx["checkpoint"]["identity"],\n               "source_provenance": ctx["source_provenance"], "full_test": ctx["run_full_test"],\n               "candidate_image_sha256": ctx["candidate_hashes"],\n               "example_image_sha256": sha(ctx["example_image"]) if ctx["example_image"] else None}\n    ctx["request_identity"] = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()\n    export_root = output / "exports"\n    export_root.mkdir(exist_ok=True)\n    name = "sun397_tacs_visual_vitl16_web_bundle" if variant == "tacs" else "sun397_mtacs_full_both_vitl16_web_bundle"\n    target = export_root / name\n    ctx.update(target=target, archive=target.with_suffix(".zip"))\n    if target.is_dir():\n        manifest = runtime.verify_bundle(target)\n        if manifest.get("export_request") != ctx["request_identity"]:\n            raise ValueError("Existing bundle belongs to a different export request")\n        ctx.update(stage=target, already_verified=True)\n        return ctx\n    if ctx["archive"].exists():\n        raise FileExistsError("Archive exists without its verified directory; preserve and inspect this output")\n    stage = export_root / (".stage_" + ctx["request_identity"][:16])\n    stage.mkdir(exist_ok=True)\n    ctx["stage"] = stage\n    if (stage / "export_request.json").is_file() and json_read(stage / "export_request.json") != request:\n        raise ValueError("Staging request changed")\n    json_write(stage / "export_request.json", request)\n    payload_bytes = sum(value.numel() * value.element_size() for value in ctx["checkpoint"]["model_state_dict"].values())\n    payload_bytes += sum(Path(row["image_path"]).stat().st_size for row in ctx["candidates"])\n    if variant == "mtacs":\n        payload_bytes += sum(path.stat().st_size for key in ("florence", "clip")\n                             for path in Path(ctx["model_assets"][key]).rglob("*") if path.is_file())\n        payload_bytes += sum((ctx["shared"] / name).stat().st_size for name in ("captions.csv", "text_embeddings.pt"))\n    already_staged = sum(path.stat().st_size for path in stage.rglob("*") if path.is_file())\n    if shutil.disk_usage(export_root).free < max(0, 2 * payload_bytes - already_staged) + 1024**3:\n        raise OSError("Not enough free disk for the remaining bundle plus ZIP. Original run is unchanged; preserve it and free other space.")\n    portable = dict(ctx["config"], candidate_count=len(ctx["candidates"]), inference_precision="float32")\n    json_write(stage / "config.json", portable)\n    json_write(stage / "class_names.json", ctx["classes"])\n    model_path = _progress(ctx, "model.pt", lambda path: common.atomic_torch_save(path, {\n        "schema": runtime.SCHEMA, "config": portable, "class_names": ctx["classes"],\n        "model_state_dict": ctx["checkpoint"]["model_state_dict"], "epoch": ctx["checkpoint"]["epoch"],\n        "run_identity": ctx["checkpoint"]["identity"], "source_best_sha256": ctx["best_sha"]}))\n    factory = ctx["model_factory"] or runtime.build_model\n    model = factory(ctx["dino_repo"], ctx["config"])\n    model.load_state_dict(ctx["checkpoint"]["model_state_dict"], strict=True)\n    model.to(ctx["device"]).eval().requires_grad_(False)\n    ctx["model"] = model\n    pool = []\n    for position, record in enumerate(common.tqdm(ctx["candidates"], desc="Copy exact saved candidate pool")):\n        name = "candidates/" + record["image_id"]\n        _copy_file(record["image_path"], visual.bundle_path(stage, name))\n        digest = sha(stage / name)\n        if digest != ctx["candidate_hashes"][record["image_id"]]:\n            raise ValueError("Candidate image changed after snapshot validation: " + record["image_id"])\n        pool.append({"position": position, "image_id": record["image_id"], "file": name,\n                     "class_id": record["class_id"], "class_name": ctx["classes"][record["class_id"]],\n                     "sha256": digest})\n    json_write(stage / "candidate_pool.json", pool)\n    common.csv_write(stage / "candidate_pool.csv", pool)\n    cfg = {**ctx["config"], "feature_batch_size": ctx["feature_batch_size"], "eval_batch_size": ctx["feature_batch_size"], "workers": ctx["workers"]}\n    ctx["eval_config"] = cfg\n\n    def feature_bank(path):\n        values = common.refresh_bank(model, ctx["candidates"], cfg, ctx["device"], amp=False)\n        common.atomic_torch_save(path, {"features": values, "image_ids": [r["image_id"] for r in pool],\n                                       "checkpoint_sha256": sha(model_path), "precision": "float32"})\n\n    bank_path = _progress(ctx, "candidate_features.pt", feature_bank)\n    bank = torch.load(bank_path, map_location="cpu", weights_only=True)\n    if (bank["image_ids"] != [r["image_id"] for r in pool] or bank["checkpoint_sha256"] != sha(model_path)\n            or tuple(bank["features"].shape) != (len(pool), ctx["config"]["feature_dim"]) or not torch.isfinite(bank["features"]).all()):\n        raise ValueError("Staged candidate bank is not aligned to this snapshot")\n    ctx["bank"] = bank["features"]\n    ctx["pool"] = pool\n    vendor = stage / "vendor/dinov3"\n    if not vendor.exists():\n        common.copy_vendor(ctx["dino_repo"], vendor)\n    for key in ("tacs_visual_web",) if variant == "tacs" else ("tacs_visual_web", "mtacs_multimodal_web", "sun397_caption_web"):\n        _copy_file(modules[key].__file__, stage / "runtime" / (key + ".py"))\n    for name in ("protocol.json", "class_names.json", "train_split.csv", "validation_split.csv", "test_split.csv", "history.csv", "training_complete.json"):\n        if (ctx["run"] / name).is_file():\n            _copy_file(ctx["run"] / name, stage / "provenance/original_run" / name)\n    for name in ("test_metrics.json", "test_predictions.csv"):\n        if (ctx["run"] / name).is_file():\n            _copy_file(ctx["run"] / name, stage / "provenance/unverified_original_metrics" / name)\n    _copy_file(__file__, stage / "provenance/export_only_source.py")\n    json_write(stage / "environment.json", common.environment())\n    json_write(stage / "export_provenance.json", {**request, "best_epoch": ctx["checkpoint"]["epoch"],\n        "training_completion": ctx["completion"], "last_saved_epoch_from_history": ctx["last_saved_epoch_from_history"],\n        "original_run": str(ctx["run"]), "candidate_order": "saved CSV; not resampled",\n        "training_called": False, "full_caption_regeneration": False,\n        "original_metrics": "reference only; not asserted to be verified by this export"})\n    if variant == "mtacs":\n        service, workflow = modules["sun397_caption_web"], modules["train_sun397_mtacs_web"]\n        texts, valid = ctx["table"].gather([r["image_id"] for r in pool], "cpu")\n        _progress(ctx, "candidate_text.pt", lambda path: common.atomic_torch_save(path, {\n            "embeddings": texts, "valid": valid, "image_ids": [r["image_id"] for r in pool],\n            "checkpoint_sha256": sha(model_path), "csv_sha256": sha(ctx["shared"] / "captions.csv")}))\n        for name in ("captions.csv", "text_embeddings.pt", "caption_protocol.json", "caption_settings.json"):\n            _copy_file(ctx["shared"] / name, stage / name)\n        for key in ("florence", "clip"):\n            _copy_assets(ctx["model_assets"][key], stage / "models" / key, service)\n            if service.tree_identity(stage / "models" / key) != ctx["model_assets"][key + "_sha256"]:\n                raise ValueError("Frozen text model changed during export: " + key)\n        (stage / "offline_wheels").mkdir(exist_ok=True)\n        if ctx["offline_assets_root"]:\n            folders = service.wheel_directories(ctx["offline_assets_root"], output / "caption_preflight/wheels")\n            for folder in folders:\n                for path in Path(folder).glob("*.whl"):\n                    if path.name.startswith(("transformers-", "tokenizers-", "huggingface_hub-", "accelerate-", "einops-", "timm-", "safetensors-")):\n                        _copy_file(path, stage / "offline_wheels" / path.name)\n        cold_source = workflow.COLD_VERIFY_SOURCE\n        requirements = "torch\\nnumpy\\nPillow\\ntqdm\\ntransformers==4.49.0\\naccelerate\\neinops\\ntimm\\n"\n    else:\n        cold_source = common.COLD_VERIFY_SOURCE\n        requirements = "torch\\nnumpy\\nPillow\\ntqdm\\n"\n    (stage / "runtime/verify_export.py").write_text(cold_source, encoding="utf-8")\n    (stage / "requirements.txt").write_text(requirements, encoding="utf-8")\n    json_write(stage / "attention_config.json", {"method": visual.ATTENTION_METHOD,\n        "query_classifier": "paired classifier CLS attention to query patches",\n        "candidate_selector": "selected candidate standalone selector CLS attention",\n        "extra_attention_model": False, "class_conditional": False, "causal_explanation": False})\n    pool_hashes = {r["sha256"] for r in pool}\n    if ctx["example_image"]:\n        example = {"image_path": str(ctx["example_image"]), "image_id": "external_export_example"}\n    else:\n        example = None\n        for record in ctx["frames"]["val"]:\n            resolved = resolve_image(record, ctx["dataset_root"])\n            if sha(resolved["image_path"]) not in pool_hashes:\n                example = resolved\n                break\n        if example is None:\n            raise ValueError("No nonduplicate held-out example; supply EXAMPLE_IMAGE explicitly")\n    example_path = stage / "examples" / ("query" + Path(example["image_path"]).suffix.lower())\n    _copy_file(example["image_path"], example_path)\n    ctx.update(example=example, example_path=example_path)\n    metrics, predictions = None, None\n    if ctx["run_full_test"]:\n        test = [resolve_image(row, ctx["dataset_root"]) for row in csv_read(ctx["run"] / "test_split.csv")]\n        if variant == "tacs":\n            metrics, predictions = common.evaluate(model, common.make_loader(test, cfg), ctx["bank"], None, ctx["candidates"], cfg, ctx["device"], label="Optional final test / snapshot", amp=False)\n        else:\n            metrics, predictions = workflow.evaluate(model, common.make_loader(test, cfg), ctx["bank"], ctx["candidates"], ctx["table"], cfg, ctx["device"], label="Optional final test / snapshot", amp=False)\n        common.csv_write(stage / "test_predictions.csv", predictions)\n    json_write(stage / "metrics.json", {"test": metrics, "test_status": "evaluated_by_export" if metrics is not None else "not_run_export_only",\n                                        "validation_from_checkpoint": ctx["checkpoint"].get("val_metrics"),\n                                        "best_epoch": ctx["checkpoint"]["epoch"], "source_best_sha256": ctx["best_sha"]})\n    (stage / "README.md").write_text(\n        "# SUN397 checkpoint snapshot web bundle\\n\\n"\n        "Exported from existing best.pt without training or an invented training_complete marker.\\n"\n        "See export_provenance.json and metrics.json: a smoke test is not a full test-set score.\\n"\n        "Full weights, exact saved candidate pool, aligned bank, runtime, DINOv3 source and real attention are included.\\n"\n        + ("Frozen Florence/CLIP assets and original caption CSV/vectors are included. Unknown images generate a fresh caption.\\n" if variant == "mtacs" else "No captioner/CLIP/text branch is used.\\n")\n        + "Use an OS/CUDA-compatible Python environment; Kaggle Linux wheels are not Windows wheels.\\n"\n        "No original Kaggle model/dataset mount is required after extraction. Keep original dataset/model license terms.\\n"\n        "Attention is last-block mean-head CLS-to-patch, not causal attribution or a separately trained model.\\n\\n"\n        + ("python runtime/mtacs_multimodal_web.py" if variant == "mtacs" else "python runtime/tacs_visual_web.py")\n        + " --bundle . --image YOUR_IMAGE.jpg --output prediction.json\\n", encoding="utf-8")\n    _manifest(ctx)\n    return ctx\n\n\ndef verify_and_publish(context):\n    ctx = context\n    stage, runtime = ctx["stage"], ctx["runtime"]\n    if not ctx.get("already_verified"):\n        production = ctx["model_factory"] is None and ctx["caption_runner"] is None\n        if production:\n            env = dict(os.environ, HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", TOKENIZERS_PARALLELISM="false")\n            env.pop("PYTHONPATH", None)\n            process = subprocess.run([sys.executable, str(stage / "runtime/verify_export.py"), "--bundle", str(stage),\n                                      "--image", str(ctx["example_path"]), "--image-id", ctx["example"]["image_id"], "--device", str(ctx["device"])],\n                                     cwd=stage, env=env, text=True, capture_output=True)\n            if process.returncode:\n                raise RuntimeError("Offline reload failed; completed export artifacts are preserved for retry:\\n" + process.stdout[-3000:] + process.stderr[-6000:])\n            prediction = json_read(stage / "examples/prediction.json")\n            data = torch.load(stage / "examples/reload.pt", map_location="cpu", weights_only=True) if ctx["variant"] == "mtacs" else {\n                "logits": torch.load(stage / "examples/reload_logits.pt", map_location="cpu", weights_only=True)}\n        else:\n            if ctx["variant"] == "tacs":\n                predictor = runtime.BundlePredictor(stage, str(ctx["device"]), ctx["model_factory"])\n                prediction = predictor.predict(ctx["example_path"], image_id=ctx["example"]["image_id"], explain=True)\n                data = {"logits": predictor.last_logits}\n            else:\n                predictor = runtime.BundlePredictor(stage, str(ctx["device"]), ctx["model_factory"], ctx["caption_runner"])\n                prediction = predictor.predict(ctx["example_path"], image_id=ctx["example"]["image_id"], explain=True, use_caption_cache=False)\n                data = {"logits": predictor.last_logits, "text": predictor.last_query_embedding, "valid": predictor.last_query_valid}\n            del predictor\n        visual = ctx["modules"]["tacs_visual_web"]\n        with torch.no_grad():\n            query = visual.preprocess(ctx["example_path"], ctx["config"])[None].to(ctx["device"])\n            excluded = [i for i, row in enumerate(ctx["pool"]) if row["image_id"] == ctx["example"]["image_id"] or row["sha256"] == sha(ctx["example_path"])]\n            if len(excluded) > 1:\n                raise ValueError("Example duplicates multiple pool images; choose another held-out example")\n            if ctx["variant"] == "tacs":\n                scores, _ = ctx["model"].score(query, ctx["bank"].to(ctx["device"]), [excluded[0] if excluded else -1])\n            else:\n                text, valid = data["text"].to(ctx["device"]), data["valid"].to(ctx["device"])\n                texts, masks = ctx["table"].gather([r["image_id"] for r in ctx["pool"]], ctx["device"])\n                scores, _ = ctx["model"].score(query, ctx["bank"].to(ctx["device"]), [excluded[0] if excluded else -1], text, valid, texts, masks)\n            position = int(scores.argmax(1))\n            if prediction["evidence"]["candidate"]["position"] != position:\n                raise ValueError("Reload context selection differs from best.pt")\n            image = visual.preprocess(ctx["candidates"][position]["image_path"], ctx["config"])[None].to(ctx["device"])\n            logits = ctx["model"](query, image) if ctx["variant"] == "tacs" else ctx["model"](query, image, text, valid, texts[position:position+1], masks[position:position+1])\n            reference = logits.float().cpu()\n        if not torch.allclose(reference, data["logits"], atol=1e-4, rtol=1e-4):\n            raise ValueError("Reload logits differ from best.pt")\n        if set(prediction["evidence"]["attention"]) != {"query_classifier", "candidate_selector"}:\n            raise ValueError("Both real attention readouts are required")\n        json_write(stage / "examples/prediction.json", prediction)\n        json_write(stage / "smoke_test.json", {"passed": True, "cold_subprocess_reload": production,\n            "fresh_florence_clip_integration": production and ctx["variant"] == "mtacs", "test_injected_services": not production,\n            "source_is_bundle_only": production,\n            "attention_roles": list(prediction["evidence"]["attention"]),\n            "max_abs_logit_error": float((reference-data["logits"]).abs().max()), "training_called": False})\n        _manifest(ctx)\n        runtime.verify_bundle(stage)\n        stage.rename(ctx["target"])\n        ctx["stage"] = ctx["target"]\n    archive = ctx["archive"]\n    if archive.is_file():\n        record = json_read(archive.parent / "download_manifest.json")\n        if record["sha256"] != sha(archive) or record["export_request"] != ctx["request_identity"]:\n            raise ValueError("Existing ZIP binding mismatch")\n        print("Reusing verified export ZIP:", archive)\n        return archive\n    temporary = archive.with_name(archive.name + ".tmp")\n    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as handle:\n        for path in sorted(ctx["target"].rglob("*")):\n            if path.is_file() and "__pycache__" not in path.parts and not path.name.endswith(".tmp"):\n                handle.write(path, ctx["target"].name + "/" + path.relative_to(ctx["target"]).as_posix())\n    with zipfile.ZipFile(temporary) as handle:\n        if handle.testzip() is not None:\n            raise ValueError("ZIP CRC check failed")\n    temporary.replace(archive)\n    json_write(archive.parent / "download_manifest.json", {"zip": archive.name, "bytes": archive.stat().st_size,\n               "sha256": sha(archive), "export_request": ctx["request_identity"], "source_best_sha256": ctx["best_sha"]})\n    print("Complete export-only ZIP:", archive)\n    return archive\n'
FALLBACK_SOURCES = {'tacs_visual_web.py': '"""Portable visual TACS model and inference contract for the SUN397 web bundle.\n\nNo captions, CLIP, label prior or external service is used. The same backbone\nencodes selector images and the concatenated query/context patch sequence.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport importlib\nimport io\nimport json\nfrom pathlib import Path, PurePosixPath\nimport sys\n\nimport numpy as np\nfrom PIL import Image, ImageOps\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.checkpoint import checkpoint\n\n\nSCHEMA = "sun397_tacs_visual_web_v1"\nATTENTION_METHOD = "last_block_cls_to_patch_attention_mean_heads"\n\n\ndef file_sha256(path):\n    with Path(path).open("rb") as handle:\n        return hashlib.file_digest(handle, "sha256").hexdigest() if hasattr(hashlib, "file_digest") else _hash_stream(handle)\n\n\ndef _hash_stream(handle):\n    digest = hashlib.sha256()\n    for chunk in iter(lambda: handle.read(4 * 1024 * 1024), b""):\n        digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef bundle_path(root, relative):\n    relative = str(relative)\n    pure = PurePosixPath(relative)\n    if pure.is_absolute() or "\\\\" in relative or ":" in relative or ".." in pure.parts:\n        raise ValueError(f"Unsafe bundle-relative path: {relative}")\n    root = Path(root).resolve()\n    target = root.joinpath(*pure.parts).resolve()\n    if not target.is_relative_to(root):\n        raise ValueError(f"Bundle path escapes root: {relative}")\n    return target\n\n\ndef read_rgb(source):\n    if isinstance(source, Image.Image):\n        return ImageOps.exif_transpose(source).convert("RGB")\n    if isinstance(source, bytes):\n        source = io.BytesIO(source)\n    with Image.open(source) as image:\n        return ImageOps.exif_transpose(image).convert("RGB")\n\n\ndef preprocess(image, config):\n    """The exported evaluation transform, including EXIF and resize semantics."""\n    image = read_rgb(image)\n    size = int(config["image_size"])\n    image = image.resize((size, size), Image.Resampling.BICUBIC)\n    pixels = torch.from_numpy(np.array(image, dtype=np.float32).copy()).permute(2, 0, 1) / 255.0\n    mean = torch.tensor(config["mean"], dtype=torch.float32)[:, None, None]\n    std = torch.tensor(config["std"], dtype=torch.float32)[:, None, None]\n    return (pixels - mean) / std\n\n\ndef concatenate_rope(rope):\n    """Token order is [query patches, context patches], not interleaved."""\n    if rope is None:\n        return None\n    if not isinstance(rope, (tuple, list)) or len(rope) != 2:\n        raise TypeError("Expected DINOv3 RoPE (sin, cos); refusing a positional fallback")\n    return tuple(torch.cat((part, part), dim=-2) for part in rope)\n\n\ndef mask_self(scores, excluded_positions=None):\n    if scores.ndim != 2 or scores.shape[1] < 2:\n        raise ValueError("TACS requires at least two candidates")\n    if excluded_positions is not None:\n        positions = torch.as_tensor(excluded_positions, dtype=torch.long, device=scores.device)\n        if positions.shape != (scores.shape[0],) or (positions >= scores.shape[1]).any():\n            raise ValueError("Invalid query-to-candidate mapping")\n        rows = torch.arange(len(positions), device=scores.device)\n        valid = positions >= 0\n        scores = scores.clone()\n        scores[rows[valid], positions[valid]] = -torch.inf\n    if not torch.isfinite(scores).any(dim=1).all():\n        raise ValueError("No eligible candidate for a query")\n    return scores\n\n\nclass _StraightThroughImage(torch.autograd.Function):\n    """Exact derivative of hard Gumbel weights @ fixed pixels, with CPU banking."""\n\n    @staticmethod\n    def forward(ctx, weights, images, chunk_size):\n        if images.ndim != 4 or weights.shape[1] != len(images):\n            raise ValueError("Image bank/selection shape mismatch")\n        hard = F.one_hot(weights.argmax(1), len(images)).to(weights.dtype)\n        if not torch.equal(weights.detach(), hard):\n            raise ValueError("Straight-through image forward requires hard one-hot values")\n        ctx.save_for_backward(images)\n        ctx.chunk_size = int(chunk_size)\n        ctx.weight_device, ctx.weight_dtype = weights.device, weights.dtype\n        indices = weights.argmax(1).to(images.device)\n        return images.index_select(0, indices).to(weights.device, dtype=torch.float32)\n\n    @staticmethod\n    def backward(ctx, grad_images):\n        (bank,) = ctx.saved_tensors\n        grad = grad_images.float().flatten(1)\n        chunks = []\n        # Float32 avoids an fp16 bank matmul silently changing selector gradients.\n        with torch.autocast(device_type=grad.device.type, enabled=False):\n            for start in range(0, len(bank), ctx.chunk_size):\n                block = bank[start:start + ctx.chunk_size].to(grad.device, dtype=torch.float32)\n                chunks.append(grad @ block.flatten(1).T)\n        return torch.cat(chunks, dim=1).to(ctx.weight_device, ctx.weight_dtype), None, None\n\n\ndef straight_through_images(weights, bank, chunk_size=32):\n    return _StraightThroughImage.apply(weights, bank, chunk_size)\n\n\nclass VisualTACS(nn.Module):\n    def __init__(self, backbone, config):\n        super().__init__()\n        self.backbone = backbone\n        self.config = dict(config)\n        dim = int(getattr(backbone, "embed_dim", config["feature_dim"]))\n        if dim != int(config["feature_dim"]):\n            raise ValueError("Backbone feature dimension does not match run configuration")\n        self.head = nn.Sequential(nn.Dropout(float(config.get("head_dropout", 0.1))), nn.Linear(dim, int(config["num_classes"])))\n        self.gradient_checkpointing = bool(config.get("gradient_checkpointing", True))\n\n    @property\n    def prefix_count(self):\n        return 1 + int(getattr(self.backbone, "n_storage_tokens", 0))\n\n    def _tokens(self, images):\n        prepared = self.backbone.prepare_tokens_with_masks(images, None)\n        if not isinstance(prepared, tuple) or len(prepared) != 2:\n            raise TypeError("This runtime requires DINOv3 prepare_tokens_with_masks -> (tokens, grid)")\n        tokens, grid = prepared\n        height, width = map(int, grid)\n        if tokens.shape[1] != self.prefix_count + height * width:\n            raise ValueError("Unexpected prefix/patch token count")\n        rope = self.backbone.rope_embed(H=height, W=width) if self.backbone.rope_embed is not None and not self.training else None\n        return tokens, rope, (height, width)\n\n    @staticmethod\n    def _cls_attention(block, tokens, rope):\n        attention = block.attn\n        qkv = attention.qkv(block.norm1(tokens))\n        batch, count, triple_dim = qkv.shape\n        dim = triple_dim // 3\n        heads = int(attention.num_heads)\n        q, k, _ = qkv.reshape(batch, count, 3, heads, dim // heads).unbind(2)\n        q, k = q.transpose(1, 2), k.transpose(1, 2)\n        if rope is not None:\n            q, k = attention.apply_rope(q, k, rope)\n        weights = ((q[:, :, :1].float() @ k.float().transpose(-2, -1)) * (dim // heads) ** -0.5).softmax(dim=-1)\n        return weights.mean(dim=1)[:, 0]\n\n    def _encode(self, tokens, rope, return_attention=False, grid=None, paired=False):\n        if return_attention and self.training:\n            raise RuntimeError("Attention export is evaluation-only")\n        cls_attention = None\n        for index, block in enumerate(self.backbone.blocks):\n            block_rope = rope\n            if self.training and self.backbone.rope_embed is not None:\n                # Native DINOv3 resamples positional augmentation per block.\n                # Pass that sample into checkpoint so backward reuses it exactly.\n                block_rope = self.backbone.rope_embed(H=grid[0], W=grid[1])\n                if paired:\n                    block_rope = concatenate_rope(block_rope)\n            if return_attention and index == len(self.backbone.blocks) - 1:\n                cls_attention = self._cls_attention(block, tokens, block_rope)\n            if self.training and self.gradient_checkpointing and torch.is_grad_enabled():\n                tokens = checkpoint(block, tokens, block_rope, use_reentrant=False, preserve_rng_state=True)\n            else:\n                tokens = block(tokens, block_rope)\n        if getattr(self.backbone, "untie_cls_and_patch_norms", False):\n            cls = self.backbone.cls_norm(tokens[:, :self.prefix_count])[:, 0]\n        else:\n            cls = self.backbone.norm(tokens)[:, 0]\n        return cls, cls_attention\n\n    def encode_selector(self, images, return_attention=False):\n        tokens, rope, grid = self._tokens(images)\n        features, attention = self._encode(tokens, rope, return_attention, grid=grid)\n        if return_attention:\n            return features, attention[:, self.prefix_count:].reshape(-1, *grid)\n        return features\n\n    def score(self, query_images, candidate_features, excluded_positions=None):\n        query = self.encode_selector(query_images).float()\n        bank = candidate_features.to(query.device, dtype=torch.float32)\n        # Eq. 1 utility: shared CLS dot product. No class labels enter scoring.\n        with torch.autocast(device_type=query.device.type, enabled=False):\n            scores = query @ bank.T\n        return mask_self(scores, excluded_positions), query\n\n    def forward(self, query_images, context_images=None, return_attention=False):\n        tokens, rope, grid = self._tokens(query_images)\n        patch_count = grid[0] * grid[1]\n        if context_images is not None:\n            context_tokens, _, context_grid = self._tokens(context_images)\n            if grid != context_grid:\n                raise ValueError("Query and context must have the same patch grid")\n            tokens = torch.cat((tokens, context_tokens[:, self.prefix_count:]), dim=1)\n            rope = concatenate_rope(rope)\n        features, attention = self._encode(tokens, rope, return_attention, grid=grid, paired=context_images is not None)\n        logits = self.head(features)\n        if not return_attention:\n            return logits\n        result = {"query_classifier": attention[:, self.prefix_count:self.prefix_count + patch_count].reshape(-1, *grid)}\n        if context_images is not None:\n            result["context_classifier"] = attention[:, self.prefix_count + patch_count:].reshape(-1, *grid)\n        return logits, result\n\n\ndef build_model(repo_dir, config, weights_path=None):\n    if config["backbone"] != "dinov3_vitl16" or int(config["feature_dim"]) != 1024:\n        raise ValueError("Production bundle requires DINOv3 ViT-L/16, 1024 dimensions; no S/B substitution")\n    repo = Path(repo_dir).resolve()\n    if not (repo / "dinov3/hub/backbones.py").is_file():\n        raise FileNotFoundError(f"DINOv3 source/dinov3/hub/backbones.py missing: {repo}")\n    kwargs = {"pretrained": weights_path is not None}\n    if weights_path is not None:\n        weights = Path(weights_path).resolve()\n        if not weights.is_file():\n            raise FileNotFoundError(weights)\n        if not weights.name.lower().endswith("lvd1689m-8aa4cbdd.pth"):\n            raise ValueError("Use the official ViT-L/16 LVD1689M 8aa4cbdd .pth with its original filename; SAT/plus formats are not supported here")\n        kwargs["weights"] = str(weights)\n    # hubconf also imports unrelated detector/segmentor dependencies. Use the\n    # exact official backbone factory without executing those entry points.\n    package = sys.modules.get("dinov3")\n    if package is not None and not Path(package.__file__).resolve().is_relative_to(repo):\n        raise RuntimeError("Another DINOv3 source is already imported. Restart the kernel before loading this bundle.")\n    sys.path.insert(0, str(repo))\n    backbones = importlib.import_module("dinov3.hub.backbones")\n    if not Path(backbones.__file__).resolve().is_relative_to(repo):\n        raise RuntimeError("DINOv3 backbone code did not load from the requested local repository")\n    backbone = backbones.dinov3_vitl16(**kwargs)\n    if len(backbone.blocks) != 24:\n        raise ValueError("Expected 24 transformer blocks for ViT-L/16")\n    return VisualTACS(backbone, config)\n\n\ndef normalize_map(raw):\n    raw = np.asarray(raw, dtype=np.float32)\n    if raw.ndim != 2 or not np.isfinite(raw).all() or raw.min() < 0:\n        raise ValueError("Invalid attention map")\n    span = float(raw.max() - raw.min())\n    return (raw - raw.min()) / max(span, 1e-12), span > 1e-12\n\n\ndef attention_overlay(image, raw, alpha=0.45):\n    image = read_rgb(image)\n    normalized, informative = normalize_map(raw)\n    heat = Image.fromarray(np.uint8(np.round(normalized * 255))).resize(image.size, Image.Resampling.BILINEAR)\n    values = np.asarray(heat, dtype=np.float32) / 255.0\n    colors = np.stack((values, 0.25 * (1 - values), 1 - values), axis=-1)\n    pixels = np.asarray(image, dtype=np.float32) / 255.0\n    mixed = (1 - alpha) * pixels + alpha * colors\n    # Flat attention is preserved and flagged, never replaced by invented structure.\n    return Image.fromarray(np.uint8(np.clip(mixed * 255, 0, 255))), informative\n\n\ndef png_data_url(image):\n    import base64\n    handle = io.BytesIO()\n    image.save(handle, format="PNG")\n    return "data:image/png;base64," + base64.b64encode(handle.getvalue()).decode("ascii")\n\n\ndef verify_bundle(root):\n    root = Path(root).resolve()\n    manifest = read_json(root / "manifest.json")\n    if manifest.get("schema") != SCHEMA or manifest.get("status") != "ready":\n        raise ValueError("Bundle is incomplete or uses another schema")\n    files = manifest.get("files", {})\n    required = {"model.pt", "config.json", "class_names.json", "candidate_pool.json", "candidate_features.pt",\n                "runtime/tacs_visual_web.py", "vendor/dinov3/dinov3/hub/backbones.py"}\n    if not required.issubset(files):\n        raise ValueError("Manifest omits a required inference artifact")\n    for name, info in files.items():\n        path = bundle_path(root, name)\n        if not path.is_file() or path.stat().st_size != int(info["bytes"]) or file_sha256(path) != info["sha256"]:\n            raise ValueError(f"Bundle checksum mismatch: {name}")\n    return manifest\n\n\nclass BundlePredictor:\n    """Load a trusted, exported bundle. No training Inputs or network are needed."""\n    def __init__(self, root, device="cpu", model_factory=build_model):\n        self.root = Path(root).resolve()\n        self.manifest = verify_bundle(self.root)\n        self.config = read_json(self.root / "config.json")\n        self.class_names = read_json(self.root / "class_names.json")\n        self.candidates = read_json(self.root / "candidate_pool.json")\n        self.device = torch.device(device)\n        if len(self.class_names) != self.config["num_classes"] or len(set(self.class_names)) != len(self.class_names):\n            raise ValueError("Class order/count is invalid")\n        if len(self.candidates) != self.config["candidate_count"]:\n            raise ValueError("Candidate count mismatch")\n        ids = [row["image_id"] for row in self.candidates]\n        if len(set(ids)) != len(ids):\n            raise ValueError("Duplicate candidate IDs")\n        for position, row in enumerate(self.candidates):\n            if row["position"] != position or row["file"] not in self.manifest["files"]:\n                raise ValueError("Candidate bank order or image manifest is invalid")\n            if row["sha256"] != self.manifest["files"][row["file"]]["sha256"]:\n                raise ValueError("Candidate content hash does not match its image")\n            if self.class_names[row["class_id"]] != row["class_name"]:\n                raise ValueError("Candidate class mapping mismatch")\n            bundle_path(self.root, row["file"])\n        checkpoint_state = torch.load(self.root / "model.pt", map_location="cpu", weights_only=True)\n        if checkpoint_state["config"] != self.config or checkpoint_state["class_names"] != self.class_names:\n            raise ValueError("Model/config/class mapping mismatch")\n        bank = torch.load(self.root / "candidate_features.pt", map_location="cpu", weights_only=True)\n        if bank["image_ids"] != ids or bank["checkpoint_sha256"] != file_sha256(self.root / "model.pt"):\n            raise ValueError("Candidate features belong to another checkpoint/order")\n        if tuple(bank["features"].shape) != (len(ids), self.config["feature_dim"]) or not torch.isfinite(bank["features"]).all():\n            raise ValueError("Candidate features are invalid")\n        self.model = model_factory(self.root / "vendor/dinov3", self.config).to(self.device)\n        self.model.load_state_dict(checkpoint_state["model_state_dict"], strict=True)\n        self.model.eval()\n        self.bank = bank["features"].float().to(self.device)\n        self.last_logits = None\n\n    def predict(self, source, top_k=5, explain=True, image_id=None):\n        image = read_rgb(source)\n        query = preprocess(image, self.config).unsqueeze(0).to(self.device)\n        excluded = torch.zeros(len(self.candidates), dtype=torch.bool, device=self.device)\n        if isinstance(source, (str, Path)):\n            content_hash = file_sha256(source)\n        elif isinstance(source, bytes):\n            content_hash = hashlib.sha256(source).hexdigest()\n        else:\n            content_hash = None\n        for i, row in enumerate(self.candidates):\n            if row["image_id"] == image_id or (content_hash is not None and row["sha256"] == content_hash):\n                excluded[i] = True\n        with torch.no_grad():\n            scores, _ = self.model.score(query, self.bank)\n            scores = scores.masked_fill(excluded.unsqueeze(0), -torch.inf)\n            if not torch.isfinite(scores).any():\n                raise ValueError("No eligible candidate after self-exclusion")\n            position = int(scores.argmax(1).item())\n            row = self.candidates[position]\n            candidate = read_rgb(bundle_path(self.root, row["file"]))\n            context = preprocess(candidate, self.config).unsqueeze(0).to(self.device)\n            if explain:\n                logits, maps = self.model(query, context, return_attention=True)\n                _, selector_map = self.model.encode_selector(context, return_attention=True)\n            else:\n                logits = self.model(query, context)\n            self.last_logits = logits.detach().cpu()\n            probabilities = logits.float().softmax(1)[0]\n            values, indices = probabilities.topk(min(max(1, int(top_k)), len(self.class_names)))\n        result = {\n            "model": "tacs", "dataset": self.config["dataset"], "bundle_schema": SCHEMA,\n            "prediction": {"class_id": int(indices[0]), "label": self.class_names[int(indices[0])], "confidence": float(values[0])},\n            "top_k": [{"class_id": int(i), "label": self.class_names[int(i)], "confidence": float(v)} for v, i in zip(values, indices)],\n            "evidence": {\n                "query_caption": {"text": None, "status": "not_used"},\n                "candidate": {"image_id": row["image_id"], "position": position, "image_url": png_data_url(candidate), "caption": None,\n                              "selector_score": float(scores[0, position]), "status": "ready"},\n                "attention": {},\n            },\n        }\n        if explain:\n            for name, raw, source_image in [("query_classifier", maps["query_classifier"][0], image),\n                                            ("candidate_selector", selector_map[0], candidate)]:\n                array = raw.float().cpu().numpy()\n                overlay, informative = attention_overlay(source_image, array)\n                result["evidence"]["attention"][name] = {\n                    "image_url": png_data_url(overlay), "status": "ready" if informative else "flat_attention",\n                    "method": ATTENTION_METHOD, "raw_grid": array.tolist(), "grid_shape": list(array.shape),\n                    "normalization": "per_image_min_max_for_display_only", "class_conditional": False,\n                }\n        return result\n\n\ndef main():\n    import argparse\n    parser = argparse.ArgumentParser(description="Offline visual TACS SUN397 bundle inference")\n    parser.add_argument("--bundle", type=Path, required=True)\n    parser.add_argument("--image", type=Path, required=True)\n    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")\n    parser.add_argument("--output", type=Path, default=Path("prediction.json"))\n    parser.add_argument("--no-attention", action="store_true")\n    args = parser.parse_args()\n    predictor = BundlePredictor(args.bundle, args.device)\n    write_json(args.output, predictor.predict(args.image, explain=not args.no_attention))\n    print(f"Saved prediction and evidence to {args.output}")\n\n\nif __name__ == "__main__":\n    main()\n', 'train_sun397_tacs_web.py': '"""SUN397 visual TACS training and verified, self-contained web export."""\nfrom __future__ import annotations\n\nimport contextlib\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nfrom pathlib import Path, PurePosixPath\nimport random\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport uuid\nimport zipfile\n\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader, Dataset\nfrom tqdm.auto import tqdm\n\nimport tacs_visual_web as runtime\n\n\ndef default_config():\n    return {\n        "schema": runtime.SCHEMA, "dataset": "SUN397", "method": "TACS_visual",\n        "backbone": "dinov3_vitl16", "feature_dim": 1024, "num_classes": 397,\n        "pretrained_variant": "LVD1689M_8aa4cbdd", "rope_training": "native_per_block_resampling",\n        "image_size": 384, "mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225],\n        "resize": "PIL_bicubic_stretch_square", "head_dropout": 0.1,\n        "context_top_k": 1, "seed": 42, "val_fraction": 0.1, "pool_fraction": 0.2,\n        "epochs": 100, "batch_size": 2, "effective_batch_size": 64, "workers": 0,\n        "feature_batch_size": 8, "eval_batch_size": 8, "st_bank_chunk": 32, "gradient_checkpointing": True,\n        "backbone_lr": 3e-6, "head_lr": 1e-4, "weight_decay": 0.05,\n        "warmup_epochs": 5, "gumbel_tau": 0.1, "policy_weight": 1.0,\n        "entropy_weight": 0.0, "label_smoothing": 0.0, "grad_clip": 1.0,\n        "utility": "shared_cls_dot_product", "candidate_bank_update": "epoch_start_and_before_validation",\n        "candidate_training_pixels": "cpu_float16_with_exact_st_weight_gradient",\n        "train_augmentation": "PIL_random_resized_crop_scale_0.7_1.0_ratio_0.75_1.333_horizontal_flip_0.5",\n        "inference_precision": "float32", "attention_method": runtime.ATTENTION_METHOD,\n        "caption_used": False, "context_label_prior": False,\n    }\n\n\ndef validate_config(config):\n    if config["backbone"] != "dinov3_vitl16" or config["feature_dim"] != 1024:\n        raise ValueError("This notebook is ViT-L/16 only")\n    if config["num_classes"] != 397 or config["context_top_k"] != 1 or config["caption_used"]:\n        raise ValueError("Expected SUN397 visual TACS, K=1, no captions")\n    if config["image_size"] < 32 or config["image_size"] % 16:\n        raise ValueError("Image size must be a positive multiple of patch size 16")\n    if config["batch_size"] < 2:\n        raise ValueError("Use train batch_size >=2 for within-batch standardized policy rewards")\n    if config["effective_batch_size"] % config["batch_size"]:\n        raise ValueError("effective_batch_size must be divisible by batch_size")\n    if config["epochs"] < 1 or not 0 < config["val_fraction"] < 1 or not 0 < config["pool_fraction"] < 1:\n        raise ValueError("Invalid epoch/split/pool settings")\n    if config["gumbel_tau"] <= 0 or config["st_bank_chunk"] < 1 or config["workers"] < 0:\n        raise ValueError("Invalid Gumbel/loader settings")\n\n\ndef seed_all(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n        torch.backends.cuda.matmul.allow_tf32 = False\n        torch.backends.cudnn.allow_tf32 = False\n        torch.backends.cudnn.benchmark = False\n\n\ndef environment():\n    versions = {}\n    for name in ("torch", "numpy", "Pillow", "tqdm", "torchvision", "transformers"):\n        try:\n            versions[name] = importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:\n            versions[name] = None\n    return {"python": sys.version, "platform": sys.platform, "packages": versions,\n            "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}\n\n\ndef csv_write(path, rows, fields=None):\n    rows = list(rows)\n    if not rows and fields is None:\n        raise ValueError("Cannot infer columns for an empty CSV")\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    with temporary.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fields or list(rows[0]), extrasaction="ignore")\n        writer.writeheader()\n        writer.writerows(rows)\n    temporary.replace(path)\n\n\ndef atomic_torch_save(path, payload):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    torch.save(payload, temporary)\n    temporary.replace(path)\n\n\ndef source_tree_identity(repo):\n    repo = Path(repo)\n    files = {p.relative_to(repo).as_posix(): runtime.file_sha256(p)\n             for p in sorted(repo.rglob("*.py")) if ".git" not in p.parts and "__pycache__" not in p.parts}\n    if "hubconf.py" not in files or not any(p.startswith("dinov3/") for p in files):\n        raise ValueError("Incomplete DINOv3 repository")\n    if not any(p.is_file() and p.name.upper().startswith(("LICENSE", "NOTICE")) for p in repo.iterdir()):\n        raise FileNotFoundError("Attach complete DINOv3 source including original LICENSE/NOTICE files before training")\n    return hashlib.sha256(json.dumps(files, sort_keys=True).encode()).hexdigest()\n\n\ndef resolve_weights(reference):\n    path = Path(reference).expanduser()\n    if path.is_file():\n        return path.resolve()\n    if path.is_dir():\n        matches = [p for p in path.rglob("*.pth") if "vitl16" in p.name.lower() and "plus" not in p.name.lower()]\n        if len(matches) == 1:\n            return matches[0].resolve()\n        raise ValueError(f"Expected one ViT-L/16 .pth in {path}; found {len(matches)}. Set the exact filename.")\n    raise FileNotFoundError(f"Attach local DINOv3 ViT-L/16 weights and set DINOV3_WEIGHTS: {path}")\n\n\ndef _relative_id(raw):\n    raw = str(raw).strip().replace("\\\\", "/").lstrip("/")\n    parts = list(PurePosixPath(raw).parts)\n    if parts and parts[0].lower() in {"sun397", "images"}:\n        parts = parts[1:]\n    if len(parts) < 3 or ".." in parts or ":" in raw:\n        raise ValueError(f"Unexpected SUN397 relative path: {raw}")\n    return PurePosixPath(*parts).as_posix()\n\n\ndef load_sun397(dataset_root, train_partition, test_partition, config):\n    root = Path(dataset_root).expanduser().resolve()\n    name_file = next((p for p in (root / "ClassName.txt", root.parent / "ClassName.txt") if p.is_file()), None)\n    if name_file is None:\n        raise FileNotFoundError(f"ClassName.txt not found at {root} or its parent")\n    classes = [line.strip().lstrip("/").replace("\\\\", "/") for line in name_file.read_text(encoding="utf-8-sig").splitlines() if line.strip()]\n    if len(classes) != config["num_classes"] or len(set(classes)) != len(classes):\n        raise ValueError("Expected 397 unique SUN397 classes, preserving ClassName.txt order")\n    class_to_id = {name: i for i, name in enumerate(classes)}\n\n    def partition(path, split):\n        path = Path(path).expanduser().resolve()\n        rows = []\n        for raw in path.read_text(encoding="utf-8-sig").splitlines():\n            if not raw.strip():\n                continue\n            image_id = _relative_id(raw)\n            class_name = str(PurePosixPath(image_id).parent)\n            if class_name not in class_to_id:\n                raise ValueError(f"Unknown class in official partition: {image_id}")\n            image_path = root.joinpath(*PurePosixPath(image_id).parts).resolve()\n            if not image_path.is_relative_to(root) or not image_path.is_file():\n                raise FileNotFoundError(f"Missing SUN397 image: {image_path}")\n            rows.append({"image_id": image_id, "image_path": str(image_path), "class_id": class_to_id[class_name],\n                         "class_name": class_name, "split": split})\n        ids = [row["image_id"] for row in rows]\n        counts = np.bincount([row["class_id"] for row in rows], minlength=len(classes))\n        if len(ids) != len(classes) * 50 or len(set(ids)) != len(ids) or not np.all(counts == 50):\n            raise ValueError(f"{path} must contain 50 distinct images per class (19,850 images)")\n        return rows\n\n    official_train = partition(train_partition, "train")\n    test = partition(test_partition, "test")\n    if {r["image_id"] for r in official_train} & {r["image_id"] for r in test}:\n        raise ValueError("Official train/test image IDs overlap")\n    rng = np.random.default_rng(config["seed"])\n    val_positions = set()\n    for class_id in range(len(classes)):\n        positions = [i for i, row in enumerate(official_train) if row["class_id"] == class_id]\n        count = max(1, min(len(positions) - 1, round(len(positions) * config["val_fraction"])))\n        val_positions.update(int(i) for i in rng.choice(positions, count, replace=False))\n    train, val = [], []\n    for i, row in enumerate(official_train):\n        item = dict(row)\n        item["split"] = "val" if i in val_positions else "train"\n        (val if i in val_positions else train).append(item)\n    count = int(len(train) * config["pool_fraction"])\n    pool_positions = np.random.default_rng(config["seed"]).choice(len(train), count, replace=False)\n    candidates = [dict(train[int(i)], position=position) for position, i in enumerate(pool_positions)]\n    if count < 2 or {r["image_id"] for r in candidates} & {r["image_id"] for r in val + test}:\n        raise ValueError("Candidate pool must be a nonempty subset of actual training images only")\n    return classes, train, val, test, candidates\n\n\ndef augment(image, config):\n    image = runtime.read_rgb(image)\n    width, height = image.size\n    area = width * height\n    box = None\n    for _ in range(10):\n        target = area * random.uniform(0.7, 1.0)\n        ratio = math.exp(random.uniform(math.log(0.75), math.log(4 / 3)))\n        crop_w, crop_h = round(math.sqrt(target * ratio)), round(math.sqrt(target / ratio))\n        if 0 < crop_w <= width and 0 < crop_h <= height:\n            x, y = random.randint(0, width - crop_w), random.randint(0, height - crop_h)\n            box = (x, y, x + crop_w, y + crop_h)\n            break\n    if box is None:\n        side = min(width, height)\n        x, y = (width - side) // 2, (height - side) // 2\n        box = (x, y, x + side, y + side)\n    image = image.crop(box)\n    if random.random() < 0.5:\n        image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)\n    return runtime.preprocess(image, config)\n\n\nclass QueryDataset(Dataset):\n    def __init__(self, rows, config, training=False):\n        self.rows, self.config, self.training = rows, config, training\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, index):\n        row = self.rows[index]\n        try:\n            image = augment(row["image_path"], self.config) if self.training else runtime.preprocess(row["image_path"], self.config)\n        except Exception as exc:\n            raise RuntimeError(f"Unreadable image {row[\'image_id\']}: {exc}") from exc\n        return {"image": image, "label": row["class_id"], "image_id": row["image_id"]}\n\n\ndef worker_seed(_worker_id):\n    value = torch.initial_seed() % (2 ** 32)\n    random.seed(value)\n    np.random.seed(value)\n\n\ndef make_loader(rows, config, training=False):\n    kwargs = {"num_workers": config["workers"], "pin_memory": torch.cuda.is_available(), "worker_init_fn": worker_seed}\n    if config["workers"]:\n        kwargs.update(persistent_workers=False, prefetch_factor=2)\n    size = config["batch_size"] if training else config["eval_batch_size"]\n    return DataLoader(QueryDataset(rows, config, training), batch_size=size, shuffle=training,\n                      drop_last=training, generator=torch.Generator().manual_seed(config["seed"]), **kwargs)\n\n\ndef build_image_bank(candidates, config):\n    count, size = len(candidates), config["image_size"]\n    bank = torch.empty((count, 3, size, size), dtype=torch.float16, device="cpu")\n    for index, row in enumerate(tqdm(candidates, desc="Candidate pixels on CPU")):\n        bank[index].copy_(runtime.preprocess(row["image_path"], config))\n    return bank\n\n\n@torch.no_grad()\ndef refresh_bank(model, candidates, config, device, image_bank=None, amp=False):\n    was_training = model.training\n    model.eval()\n    chunks = []\n    for start in range(0, len(candidates), config["feature_batch_size"]):\n        group = candidates[start:start + config["feature_batch_size"]]\n        # Evaluate raw images in fp32; the optional fp16 pixel bank is training-only.\n        images = torch.stack([runtime.preprocess(r["image_path"], config) for r in group]).to(device)\n        with autocast_context(device, amp):\n            features = model.encode_selector(images)\n        chunks.append(features.detach().float().cpu())\n    model.train(was_training)\n    bank = torch.cat(chunks)\n    if not torch.isfinite(bank).all():\n        raise FloatingPointError("Candidate features contain NaN/Inf")\n    return bank\n\n\ndef autocast_context(device, enabled=True):\n    device = torch.device(device)\n    if device.type != "cuda" or not enabled:\n        return contextlib.nullcontext()\n    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n    return torch.autocast("cuda", dtype=dtype)\n\n\ndef classification_metrics(truth, predictions, classes):\n    counts = np.zeros((classes, classes), dtype=np.int64)\n    np.add.at(counts, (np.asarray(truth, dtype=int), np.asarray(predictions, dtype=int)), 1)\n    tp = counts.diagonal()\n    denominator = counts.sum(0) + counts.sum(1)\n    f1 = np.divide(2 * tp, denominator, out=np.zeros(classes, dtype=float), where=denominator != 0)\n    return {"accuracy": float(tp.sum() / max(1, counts.sum())), "macro_f1": float(f1.mean())}\n\n\ndef rng_state(loader):\n    numpy_state = np.random.get_state()\n    return {"python": random.getstate(), "numpy": [numpy_state[0], numpy_state[1].tolist(), numpy_state[2], numpy_state[3], numpy_state[4]],\n            "torch": torch.get_rng_state(), "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],\n            "loader": loader.generator.get_state()}\n\n\ndef restore_rng(saved, loader):\n    random.setstate(saved["python"])\n    state = saved["numpy"]\n    np.random.set_state((state[0], np.asarray(state[1], dtype=np.uint32), state[2], state[3], state[4]))\n    torch.set_rng_state(saved["torch"].cpu())\n    if torch.cuda.is_available() and saved["cuda"]:\n        torch.cuda.set_rng_state_all([s.cpu() for s in saved["cuda"]])\n    loader.generator.set_state(saved["loader"].cpu())\n\n\ndef run_identity(config, classes, frames, assets):\n    ids = {key: [(r["image_id"], int(r["class_id"])) for r in value] for key, value in frames.items()}\n    payload = {"config": config, "classes": classes, "splits_and_pool": ids, "assets": assets,\n               "model_source_sha256": runtime.file_sha256(Path(runtime.__file__)), "training_source_sha256": runtime.file_sha256(__file__)}\n    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()\n\n\ndef train_epoch(model, loader, feature_bank, pixel_bank, candidates, optimizer, scaler, config, device, epoch):\n    model.train()\n    bank = feature_bank.to(device)\n    positions = {row["image_id"]: i for i, row in enumerate(candidates)}\n    accumulate = config["effective_batch_size"] // config["batch_size"]\n    stats = {key: 0.0 for key in ("loss", "ce_loss", "policy_loss", "reward", "entropy")}\n    truth, predictions = [], []\n    optimizer_steps = skipped_steps = 0\n    optimizer.zero_grad(set_to_none=True)\n    steps = len(loader)\n    if steps < 1:\n        raise ValueError("Training loader is empty")\n    for step, batch in enumerate(tqdm(loader, desc=f"Epoch {epoch:03d} train")):\n        query, labels = batch["image"].to(device), batch["label"].to(device)\n        excluded = torch.tensor([positions.get(i, -1) for i in batch["image_id"]], device=device)\n        with autocast_context(device):\n            scores, _ = model.score(query, bank, excluded)\n            weights = F.gumbel_softmax(scores, tau=config["gumbel_tau"], hard=True, dim=1)\n            context = runtime.straight_through_images(weights, pixel_bank, config["st_bank_chunk"])\n            logits = model(query, context)\n            ce = F.cross_entropy(logits.float(), labels, label_smoothing=config["label_smoothing"])\n            distribution = torch.distributions.Categorical(logits=scores)\n            actions = distribution.sample()\n            log_probs = distribution.log_prob(actions)\n            entropy = distribution.entropy().mean()\n        # Reward passes use the same eval-mode task network and detached outputs.\n        model.eval()\n        with torch.no_grad(), autocast_context(device):\n            baseline = model(query)\n            reward_context = pixel_bank.index_select(0, actions.cpu()).to(device, dtype=torch.float32)\n            paired = model(query, reward_context)\n            reward = (F.cross_entropy(baseline.float(), labels, reduction="none") -\n                      F.cross_entropy(paired.float(), labels, reduction="none")).detach()\n            advantage = ((reward - reward.mean()) / reward.std(unbiased=False).clamp_min(1e-6)).detach()\n        model.train()\n        policy = -(log_probs * advantage).mean()\n        loss = ce + config["policy_weight"] * policy - config["entropy_weight"] * entropy\n        if not torch.isfinite(loss):\n            raise FloatingPointError(f"Nonfinite loss at epoch {epoch}, batch {step}")\n        group_start = (step // accumulate) * accumulate\n        divisor = min(accumulate, steps - group_start)\n        scaler.scale(loss / divisor).backward()\n        if (step + 1) % accumulate == 0 or step + 1 == steps:\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"], error_if_nonfinite=not scaler.is_enabled())\n            previous_scale = scaler.get_scale()\n            scaler.step(optimizer)\n            scaler.update()\n            skipped = scaler.is_enabled() and scaler.get_scale() < previous_scale\n            skipped_steps += int(skipped)\n            optimizer_steps += int(not skipped)\n            optimizer.zero_grad(set_to_none=True)\n        count = len(labels)\n        for key, value in (("loss", loss), ("ce_loss", ce), ("policy_loss", policy), ("reward", reward.mean()), ("entropy", entropy)):\n            stats[key] += float(value.detach()) * count\n        truth.extend(labels.cpu().tolist())\n        predictions.extend(logits.detach().argmax(1).cpu().tolist())\n    if not optimizer_steps:\n        raise FloatingPointError("Every optimizer step overflowed; inspect AMP/learning rates before resuming")\n    return {**{k: v / len(truth) for k, v in stats.items()}, **classification_metrics(truth, predictions, config["num_classes"]),\n            "optimizer_steps": optimizer_steps, "amp_skipped_steps": skipped_steps}\n\n\n@torch.no_grad()\ndef evaluate(model, loader, bank, pixel_bank, candidates, config, device, label="validation", amp=False):\n    model.eval()\n    features = bank.to(device)\n    positions = {row["image_id"]: i for i, row in enumerate(candidates)}\n    rows, loss_sum = [], 0.0\n    for batch in tqdm(loader, desc=label):\n        query, labels = batch["image"].to(device), batch["label"].to(device)\n        excluded = torch.tensor([positions.get(i, -1) for i in batch["image_id"]], device=device)\n        with autocast_context(device, amp):\n            scores, _ = model.score(query, features, excluded)\n            indices = scores.argmax(1)\n            # Raw files + identical eval transform, matching exported inference.\n            context = torch.stack([runtime.preprocess(candidates[int(i)]["image_path"], config) for i in indices.cpu()]).to(device)\n            logits = model(query, context)\n            loss_sum += float(F.cross_entropy(logits.float(), labels, reduction="sum"))\n        for j, image_id in enumerate(batch["image_id"]):\n            pos = int(indices[j])\n            rows.append({"image_id": image_id, "ground_truth": int(labels[j]), "prediction": int(logits[j].argmax()),\n                         "candidate_position": pos, "candidate_id": candidates[pos]["image_id"]})\n    if not rows:\n        raise ValueError("Evaluation loader is empty")\n    metrics = classification_metrics([r["ground_truth"] for r in rows], [r["prediction"] for r in rows], config["num_classes"])\n    return {**metrics, "loss": loss_sum / len(rows), "images": len(rows)}, rows\n\n\ndef train_run(model, classes, train, val, candidates, config, assets, output, device):\n    validate_config(config)\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    identity = run_identity(config, classes, {"train": train, "val": val, "pool": candidates}, assets)\n    protocol = output / "protocol.json"\n    if protocol.is_file() and runtime.read_json(protocol)["identity"] != identity:\n        raise ValueError("Output directory belongs to another run. Do not change seed/epochs/pool/source on resume.")\n    source_hashes = {"runtime": runtime.file_sha256(runtime.__file__), "training": runtime.file_sha256(__file__)}\n    runtime.write_json(protocol, {"identity": identity, "config": config, "assets": assets,\n                                  "source_sha256": source_hashes, "environment": environment()})\n    csv_write(output / "train_split.csv", train)\n    csv_write(output / "validation_split.csv", val)\n    csv_write(output / "candidate_pool.csv", candidates)\n    runtime.write_json(output / "class_names.json", classes)\n    train_loader, val_loader = make_loader(train, config, True), make_loader(val, config)\n    pixel_bank = build_image_bank(candidates, config)\n    optimizer = torch.optim.AdamW([\n        {"params": model.backbone.parameters(), "lr": config["backbone_lr"]},\n        {"params": model.head.parameters(), "lr": config["head_lr"]},\n    ], weight_decay=config["weight_decay"])\n\n    def lr_scale(epoch):\n        warmup = min(config["warmup_epochs"], config["epochs"])\n        if warmup and epoch < warmup:\n            return 0.2 + 0.8 * epoch / warmup\n        progress = (epoch - warmup) / max(1, config["epochs"] - warmup)\n        return 0.01 + 0.99 * (1 + math.cos(math.pi * min(1, progress))) / 2\n\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_scale)\n    scaler = torch.amp.GradScaler("cuda", enabled=torch.device(device).type == "cuda" and not torch.cuda.is_bf16_supported())\n    last_path, best_path = output / "last.pt", output / "best.pt"\n    history, best_loss, start_epoch = [], math.inf, 1\n    if last_path.is_file():\n        last = torch.load(last_path, map_location="cpu", weights_only=True)\n        if last["identity"] != identity:\n            raise ValueError("Resume identity mismatch")\n        model.load_state_dict(last["model_state_dict"], strict=True)\n        optimizer.load_state_dict(last["optimizer_state_dict"])\n        scheduler.load_state_dict(last["scheduler_state_dict"])\n        scaler.load_state_dict(last["scaler_state_dict"])\n        restore_rng(last["rng"], train_loader)\n        history, best_loss, start_epoch = last["history"], last["best_val_loss"], last["epoch"] + 1\n        del last\n        if not best_path.is_file():\n            raise FileNotFoundError("Resume requires this run\'s best.pt as well as last.pt")\n        print(f"Resuming after completed epoch {start_epoch - 1}; horizon stays {config[\'epochs\']}")\n    for epoch in range(start_epoch, config["epochs"] + 1):\n        started = time.perf_counter()\n        bank = refresh_bank(model, candidates, config, device, amp=True)\n        train_metrics = train_epoch(model, train_loader, bank, pixel_bank, candidates, optimizer, scaler, config, device, epoch)\n        bank = refresh_bank(model, candidates, config, device, amp=True)\n        val_metrics, _ = evaluate(model, val_loader, bank, pixel_bank, candidates, config, device, amp=True)\n        improved = val_metrics["loss"] < best_loss\n        best_loss = min(best_loss, val_metrics["loss"])\n        history.append({"epoch": epoch, **{"train_" + k: v for k, v in train_metrics.items()},\n                        **{"val_" + k: v for k, v in val_metrics.items()}, "seconds": time.perf_counter() - started,\n                        "backbone_lr": optimizer.param_groups[0]["lr"]})\n        scheduler.step()\n        model_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}\n        base = {"schema": runtime.SCHEMA, "identity": identity, "epoch": epoch, "model_state_dict": model_state,\n                "config": config, "class_names": classes, "val_metrics": val_metrics}\n        if improved:\n            atomic_torch_save(best_path, base)\n        atomic_torch_save(last_path, {**base, "optimizer_state_dict": optimizer.state_dict(), "scheduler_state_dict": scheduler.state_dict(),\n                                     "scaler_state_dict": scaler.state_dict(), "rng": rng_state(train_loader),\n                                     "history": history, "best_val_loss": best_loss})\n        csv_write(output / "history.csv", history)\n        print(f"epoch={epoch:03d}/{config[\'epochs\']} val_loss={val_metrics[\'loss\']:.6f} val_f1={val_metrics[\'macro_f1\']:.6f} best={improved}")\n        del model_state, base\n    if not history or history[-1]["epoch"] != config["epochs"]:\n        raise RuntimeError("Training budget did not complete; final test/export are blocked")\n    best = torch.load(best_path, map_location="cpu", weights_only=True)\n    if best["identity"] != identity:\n        raise ValueError("Best checkpoint belongs to another run")\n    model.load_state_dict(best["model_state_dict"], strict=True)\n    model.eval()\n    runtime.write_json(output / "training_complete.json", {"identity": identity, "epochs_completed": config["epochs"],\n                                                         "best_epoch": best["epoch"], "best_sha256": runtime.file_sha256(best_path)})\n    return best, history\n\n\ndef copy_vendor(repo, destination):\n    repo, destination = Path(repo), Path(destination)\n    package = repo / "dinov3"\n    if not package.is_dir() or not (repo / "hubconf.py").is_file():\n        raise FileNotFoundError("Cannot export incomplete DINOv3 source")\n    ignore = shutil.ignore_patterns("__pycache__", "*.pyc", ".git", "*.pth", "*.pt", "*.safetensors", ".cache")\n    destination.mkdir(parents=True, exist_ok=False)\n    # Backbone inference needs these upstream modules, not long training YAML\n    # trees or detection/segmentation imports. Preserve selected sources verbatim.\n    vendored = destination / "dinov3"\n    vendored.mkdir()\n    shutil.copy2(package / "__init__.py", vendored / "__init__.py")\n    for name in ("models", "layers", "utils"):\n        shutil.copytree(package / name, vendored / name, ignore=ignore)\n    (vendored / "hub").mkdir()\n    for name in ("__init__.py", "backbones.py", "utils.py"):\n        shutil.copy2(package / "hub" / name, vendored / "hub" / name)\n    licenses = [p for p in repo.iterdir() if p.is_file() and (p.name.upper().startswith(("LICENSE", "NOTICE")))]\n    if not licenses:\n        raise FileNotFoundError("Attach the original DINOv3 source with its license/notice files")\n    for path in licenses:\n        shutil.copy2(path, destination / path.name)\n\n\ndef export_bundle(model, best, candidates, classes, config, repo, output, val_example, test_metrics, test_predictions,\n                  model_factory=runtime.build_model, require_training_complete=True):\n    """Only call after loading best.pt; always rebuild the final feature bank."""\n    output = Path(output).resolve()\n    if require_training_complete:\n        complete = runtime.read_json(output / "training_complete.json")\n        if complete["identity"] != best["identity"] or complete["best_epoch"] != best["epoch"]:\n            raise ValueError("Training/best/export identity mismatch")\n        if complete["best_sha256"] != runtime.file_sha256(output / "best.pt"):\n            raise ValueError("Best checkpoint changed after completion")\n    if classes != best["class_names"] or config != best["config"]:\n        raise ValueError("Export arguments do not match the best checkpoint")\n    protocol = runtime.read_json(output / "protocol.json")\n    if protocol["identity"] != best["identity"] or protocol["config"] != config:\n        raise ValueError("Export protocol does not match the trained run")\n    source_hashes = {"runtime": runtime.file_sha256(runtime.__file__), "training": runtime.file_sha256(__file__)}\n    if protocol["source_sha256"] != source_hashes:\n        raise ValueError("Runtime/training source changed after training")\n    expected_source = protocol["assets"].get("dino_source_sha256")\n    if expected_source is not None and source_tree_identity(repo) != expected_source:\n        raise ValueError("Export DINOv3 source differs from the training source")\n    with (output / "candidate_pool.csv").open(encoding="utf-8", newline="") as handle:\n        trained_pool = [(r["image_id"], int(r["class_id"])) for r in csv.DictReader(handle)]\n    if trained_pool != [(r["image_id"], int(r["class_id"])) for r in candidates]:\n        raise ValueError("Export candidate pool/order differs from the trained pool")\n    model.load_state_dict(best["model_state_dict"], strict=True)\n    model.eval()\n    device = next(model.parameters()).device\n    export_root = output / "exports"\n    export_root.mkdir(parents=True, exist_ok=True)\n    target = export_root / "sun397_tacs_visual_vitl16_web_bundle"\n    archive = target.with_suffix(".zip")\n    best_sha = runtime.file_sha256(output / "best.pt")\n    if target.exists() or archive.exists():\n        if target.is_dir() and archive.is_file() and (export_root / "download_manifest.json").is_file():\n            saved_manifest = runtime.verify_bundle(target)\n            download = runtime.read_json(export_root / "download_manifest.json")\n            if (saved_manifest["run_identity"] == best["identity"]\n                    and saved_manifest.get("training_checkpoint_sha256") == best_sha\n                    and download["zip"] == archive.name and download["bytes"] == archive.stat().st_size\n                    and download["sha256"] == runtime.file_sha256(archive)):\n                print(f"Reusing this run\'s verified deployment ZIP: {archive}")\n                return archive\n        raise FileExistsError("Export target exists but cannot be verified as this run\'s complete bundle. Preserve it and inspect exports/.")\n    parameter_bytes = sum(v.numel() * v.element_size() for v in model.state_dict().values())\n    image_bytes = sum(Path(row["image_path"]).stat().st_size for row in candidates)\n    if shutil.disk_usage(export_root).free < 2 * (parameter_bytes + image_bytes) + 1024 ** 3:\n        raise OSError("Not enough disk space for bundle plus ZIP; preserve/move last.pt before exporting")\n    stage = export_root / (".staging_" + uuid.uuid4().hex)\n    stage.mkdir()\n    portable_config = dict(config, candidate_count=len(candidates), inference_precision="float32")\n    runtime.write_json(stage / "config.json", portable_config)\n    runtime.write_json(stage / "class_names.json", classes)\n    atomic_torch_save(stage / "model.pt", {"schema": runtime.SCHEMA, "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},\n                                          "class_names": classes, "config": portable_config, "epoch": best["epoch"], "run_identity": best["identity"]})\n    checkpoint_sha = runtime.file_sha256(stage / "model.pt")\n    pool = []\n    for position, row in enumerate(tqdm(candidates, desc="Bundle candidate images")):\n        name = "candidates/" + row["image_id"]\n        destination = runtime.bundle_path(stage, name)\n        destination.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copy2(row["image_path"], destination)\n        pool.append({"position": position, "image_id": row["image_id"], "file": name, "class_id": int(row["class_id"]),\n                     "class_name": classes[int(row["class_id"])], "sha256": runtime.file_sha256(destination)})\n    runtime.write_json(stage / "candidate_pool.json", pool)\n    csv_write(stage / "candidate_pool.csv", pool)\n    final_bank = refresh_bank(model, candidates, config, device, amp=False)\n    atomic_torch_save(stage / "candidate_features.pt", {"features": final_bank, "image_ids": [r["image_id"] for r in pool],\n                                                      "checkpoint_sha256": checkpoint_sha, "precision": "float32"})\n    copy_vendor(repo, stage / "vendor/dinov3")\n    (stage / "runtime").mkdir()\n    shutil.copy2(Path(runtime.__file__), stage / "runtime/tacs_visual_web.py")\n    (stage / "provenance").mkdir()\n    shutil.copy2(Path(__file__), stage / "provenance/training_source.py")\n    runtime.write_json(stage / "environment.json", environment())\n    runtime.write_json(stage / "metrics.json", {"test": test_metrics, "validation_best": best["val_metrics"], "best_epoch": best["epoch"],\n                                              "run_identity": best["identity"], "test_precision": "float32"})\n    csv_write(stage / "test_predictions.csv", test_predictions)\n    if (output / "history.csv").is_file():\n        shutil.copy2(output / "history.csv", stage / "history.csv")\n    shutil.copy2(output / "protocol.json", stage / "protocol.json")\n    for name in ("train_split.csv", "validation_split.csv", "test_split.csv"):\n        if (output / name).is_file():\n            shutil.copy2(output / name, stage / "provenance" / name)\n    (stage / "examples").mkdir()\n    example_path = stage / "examples" / ("query" + Path(val_example["image_path"]).suffix.lower())\n    shutil.copy2(val_example["image_path"], example_path)\n    runtime.write_json(stage / "attention_config.json", {\n        "method": runtime.ATTENTION_METHOD, "layer": "last_transformer_block", "heads": "mean",\n        "query_classifier": "query patch slice of pair-input classifier CLS attention",\n        "candidate_selector": "selected candidate standalone selector-encoder CLS attention",\n        "raw_maps_included": True, "class_conditional": False, "causal_explanation": False,\n        "no_context_comparison_model_included": False,\n        "note": "No separate attention network is needed; maps are recomputed by the trained backbone. Not an exact replication of every Figure 4 visualization choice.",\n    })\n    versions = environment()["packages"]\n    requirements = [f"{name}=={versions[name].split(\'+\')[0]}" for name in ("torch", "numpy", "Pillow", "tqdm")]\n    (stage / "requirements.txt").write_text("\\n".join(requirements) + "\\n", encoding="utf-8")\n    (stage / "runtime/verify_export.py").write_text(COLD_VERIFY_SOURCE, encoding="utf-8")\n    (stage / "README.md").write_text(\n        "# SUN397 visual TACS ViT-L/16 web bundle\\n\\n"\n        "Full trained weights, exact class order, candidate images and bank, vendored DINOv3 backbone code and runtime.\\n"\n        "No SUN397 training mount or pretrained weight file is needed after extraction.\\n"\n        "Install matching PyTorch for your OS/CUDA and the listed dependencies in a separate environment.\\n"\n        "No Florence, CLIP or captions are used in this visual bundle.\\n\\n"\n        "```bash\\npython runtime/tacs_visual_web.py --bundle . --image examples/" + example_path.name + " --output prediction.json\\n```\\n\\n"\n        "For FastAPI, import BundlePredictor from runtime/tacs_visual_web.py, load once and call predict(file_bytes).\\n"\n        "It returns model/prediction/top_k and evidence.candidate plus query_classifier/candidate_selector PNG data URLs.\\n"\n        "Do not call the CPU/GPU predictor concurrently without a lock. API wiring in the existing web is a separate step.\\n"\n        "Attention is last-block mean-head CLS attention, not Grad-CAM or a class-conditioned causal claim.\\n"\n        "There is no separately trained no-context model, text captioner or textual feature naming in this bundle.\\n"\n        "Keep original DINOv3/model/dataset license terms; do not assume public redistribution rights.\\n",\n        encoding="utf-8")\n    # The manifest is refreshed after offline reload/parity and explanation checks.\n    def manifest():\n        files = {p.relative_to(stage).as_posix(): {"bytes": p.stat().st_size, "sha256": runtime.file_sha256(p)}\n                 for p in sorted(stage.rglob("*")) if p.is_file() and p.name != "manifest.json" and "__pycache__" not in p.parts}\n        runtime.write_json(stage / "manifest.json", {"schema": runtime.SCHEMA, "status": "ready", "files": files,\n                                                   "run_identity": best["identity"], "model_sha256": checkpoint_sha,\n                                                   "training_checkpoint_sha256": best_sha})\n    manifest()\n    cold_reload = model_factory is runtime.build_model\n    if cold_reload:\n        env = dict(os.environ, HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", TOKENIZERS_PARALLELISM="false")\n        env.pop("PYTHONPATH", None)\n        process = subprocess.run([sys.executable, str(stage / "runtime/verify_export.py"), "--bundle", str(stage),\n                                  "--image", str(example_path), "--image-id", val_example["image_id"], "--device", str(device)],\n                                 cwd=stage, env=env, text=True, capture_output=True)\n        if process.returncode:\n            raise RuntimeError("Cold offline bundle reload failed:\\n" + process.stdout[-4000:] + process.stderr[-8000:])\n        result = runtime.read_json(stage / "examples/prediction.json")\n        reloaded_logits = torch.load(stage / "examples/reload_logits.pt", map_location="cpu", weights_only=True)\n    else:\n        # Dependency injection is used by CPU unit tests, never the notebook\'s production export.\n        predictor = runtime.BundlePredictor(stage, device=str(device), model_factory=model_factory)\n        result = predictor.predict(example_path, explain=True, image_id=val_example["image_id"])\n        reloaded_logits = predictor.last_logits\n        del predictor\n    position = result["evidence"]["candidate"]["position"]\n    with torch.no_grad():\n        query = runtime.preprocess(example_path, config)[None].to(device)\n        reference_scores, _ = model.score(query, final_bank.to(device))\n        example_sha = runtime.file_sha256(example_path)\n        for i, row in enumerate(pool):\n            if row["sha256"] == example_sha or row["image_id"] == val_example["image_id"]:\n                reference_scores[:, i] = -torch.inf\n        if int(reference_scores.argmax(1)) != position:\n            raise ValueError("Export reload selected another candidate than the best checkpoint")\n        reference = model(query, runtime.preprocess(candidates[position]["image_path"], config)[None].to(device)).float().cpu()\n    error = float((reference - reloaded_logits).abs().max())\n    if not torch.allclose(reference, reloaded_logits, atol=1e-4, rtol=1e-4):\n        raise ValueError(f"Export reload logits disagree with best checkpoint: max_abs={error}")\n    for role, item in result["evidence"]["attention"].items():\n        if item["method"] != runtime.ATTENTION_METHOD or not np.isfinite(np.asarray(item["raw_grid"])).all():\n            raise ValueError(f"Invalid exported attention: {role}")\n    runtime.write_json(stage / "examples/prediction.json", result)\n    runtime.write_json(stage / "smoke_test.json", {"passed": True, "offline_local_reload": True, "cold_subprocess_reload": cold_reload,\n                                                  "strict_state_dict": True,\n                                                  "max_abs_logit_error": error, "query_image_id": val_example["image_id"],\n                                                  "selected_candidate_id": result["evidence"]["candidate"]["image_id"],\n                                                  "attention_roles": list(result["evidence"]["attention"]),\n                                                  "flat_attention_roles": [k for k, v in result["evidence"]["attention"].items() if v["status"] != "ready"]})\n    manifest()\n    runtime.verify_bundle(stage)\n    stage.rename(target)\n    temporary = archive.with_name(archive.name + ".tmp")\n    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as handle:\n        for path in sorted(target.rglob("*")):\n            if path.is_file() and "__pycache__" not in path.parts:\n                handle.write(path, arcname=target.name + "/" + path.relative_to(target).as_posix())\n    with zipfile.ZipFile(temporary) as handle:\n        bad = handle.testzip()\n        if bad:\n            raise ValueError(f"Corrupt ZIP member: {bad}")\n    temporary.replace(archive)\n    runtime.write_json(export_root / "download_manifest.json", {"zip": archive.name, "bytes": archive.stat().st_size,\n                                                              "sha256": runtime.file_sha256(archive), "bundle": target.name,\n                                                              "model": config["backbone"], "schema": runtime.SCHEMA})\n    print(f"Verified deployment ZIP: {archive}")\n    return archive\n\n\nCOLD_VERIFY_SOURCE = \'\'\'"""Cold-process inference check using only extracted bundle assets."""\nimport argparse\nimport sys\nfrom pathlib import Path\nimport torch\nfrom tacs_visual_web import BundlePredictor, write_json\n\nparser = argparse.ArgumentParser()\nparser.add_argument("--bundle", type=Path, required=True)\nparser.add_argument("--image", type=Path, required=True)\nparser.add_argument("--image-id", required=True)\nparser.add_argument("--device", required=True)\nargs = parser.parse_args()\nif torch.cuda.is_available():\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\npredictor = BundlePredictor(args.bundle, args.device)\nimport dinov3\nif not Path(dinov3.__file__).resolve().is_relative_to((args.bundle / "vendor/dinov3").resolve()):\n    raise RuntimeError("Reload used an external DINOv3 source instead of the bundle")\nplain = predictor.predict(args.image, explain=False, image_id=args.image_id)\nplain_logits = predictor.last_logits.clone()\nexplained = predictor.predict(args.image, explain=True, image_id=args.image_id)\nif plain["evidence"]["candidate"]["image_id"] != explained["evidence"]["candidate"]["image_id"]:\n    raise RuntimeError("Explanation changed context selection")\nif not torch.allclose(plain_logits, predictor.last_logits, atol=1e-5, rtol=1e-5):\n    raise RuntimeError("Attention capture changed model logits")\nwrite_json(args.bundle / "examples/prediction.json", explained)\ntorch.save(predictor.last_logits, args.bundle / "examples/reload_logits.pt")\nprint("Cold bundle inference, both real attention maps and logits parity passed")\n\'\'\'\n'}
exporter_path = EXPORT_CODE_DIR / "sun397_export_only.py"
if exporter_path.is_file() and exporter_path.read_text(encoding="utf-8") != EXPORTER_SOURCE:
    raise ValueError("Exporter source changed under this output; preserve it and choose a new OUTPUT_DIR")
if not exporter_path.is_file():
    exporter_path.write_text(EXPORTER_SOURCE, encoding="utf-8")
existing = sys.modules.get("sun397_export_only")
if existing is not None and Path(existing.__file__).resolve() != exporter_path.resolve():
    raise RuntimeError("Another exporter is imported; restart the kernel")
sys.path.insert(0, str(EXPORT_CODE_DIR.resolve()))
import sun397_export_only as exporter


## 4 Validate snapshot, original pool and source identity

This checks checkpoint/config/class order, the saved split/pool identity, the original DINO source, and the exact inference runtime. It does not need last.pt or a completion marker. A present but mismatched completion marker is an error, not something to overwrite. M-TACS additionally validates original caption hashes, masks, image bindings and frozen text assets. No random replacement pool, substitute captions or partial state_dict load is allowed.


In [ ]:
CONTEXT = exporter.prepare(
    SOURCE_RUN_DIR, DATASET_ROOT, DINOV3_REPO_DIR, OUTPUT_DIR, VARIANT, FALLBACK_SOURCES,
    device=DEVICE, feature_batch_size=FEATURE_BATCH_SIZE, workers=NUM_WORKERS,
    example_image=EXAMPLE_IMAGE, run_full_test=RUN_FULL_TEST,
)
display({"variant": CONTEXT["variant"], "best_epoch": CONTEXT["checkpoint"]["epoch"],
         "candidate_count": len(CONTEXT["candidates"]), "completion": CONTEXT["completion"],
         "run_full_test": RUN_FULL_TEST})


## 5 Build the missing model and bank artifacts

Strictly load the full best weights without a pretrained DINO .pth, copy only the saved pool, encode its images once in fp32, and bind every feature to model.pt and exact image order. Verified completed stages are reused on retry. No optimizer or training loop is involved. M-TACS slices the original CLIP cache for candidate_text.pt, copies the existing CSV/vectors and complete frozen caption models. It never calls full caption regeneration.

If RUN_FULL_TEST=False, metrics.json explicitly says not_run_export_only; it does not invent scores. Optional full test uses the saved test split and current snapshot, not test-time training. Training completion is not fabricated for export.


In [ ]:
CONTEXT = exporter.build(CONTEXT)
print("Bundle directory (staging until verified):", CONTEXT["stage"])
display(exporter.json_read(CONTEXT["stage"] / "metrics.json"))


## 6 Cold reload, actual attention and ZIP

A fresh process loads only the staged bundle. It verifies logits with/without attention and matches the original loaded best checkpoint. M-TACS also forces one fresh Florence/CLIP caption using the bundled weights, so novel-image inference is exercised. Failures preserve the staged files for inspection/retry; an incomplete bundle is not published as ready.


In [ ]:
EXPORT_ZIP = exporter.verify_and_publish(CONTEXT)
print("Download:", EXPORT_ZIP)


## 7 Inspect the exported files and evidence
Download the complete ZIP, not only model.pt. Keep the original best/run folder separately for provenance or future training resume.


In [ ]:
import base64
from IPython.display import FileLink, Image as DisplayImage, display
BUNDLE = EXPORT_ZIP.with_suffix("")
for name in ("model.pt", "candidate_features.pt", "candidate_text.pt"):
    path = BUNDLE / name
    if path.is_file():
        print(name, "|", round(path.stat().st_size / 1024**2, 2), "MiB")
display(exporter.json_read(BUNDLE / "smoke_test.json"))
display(exporter.json_read(BUNDLE / "metrics.json"))
result = exporter.json_read(BUNDLE / "examples/prediction.json")
display(result["prediction"])
display(result["evidence"]["query_caption"])
display(result["evidence"]["candidate"] | {"image_url": "shown below"})
display(DisplayImage(data=base64.b64decode(result["evidence"]["candidate"]["image_url"].split(",", 1)[1])))
for role, value in result["evidence"]["attention"].items():
    print(role, "|", value["method"], "|", value["status"])
    display(DisplayImage(data=base64.b64decode(value["image_url"].split(",", 1)[1])))
display(FileLink(str(EXPORT_ZIP)))
